# v18 — v17 고정 기준과 트리 다양성 실험
train 전용. test/sample 접근 차단, 기존 홀드아웃 미평가. 이전 개발 분할을 재사용하므로 독립 검증이 아닙니다. 내부 선택과 바깥 평가를 분리하고, 사전 기준을 통과한 경우 새 seed에서 고정 조합을 확인합니다. 노트북만으로 선별·학습·저장합니다.

In [1]:
import os,sys,re,json,pickle,cloudpickle,hashlib,platform,time,copy,warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.special import softmax
from sklearn.base import BaseEstimator,TransformerMixin,clone
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_selection import chi2,f_classif
from sklearn.preprocessing import StandardScaler,MaxAbsScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix
from lightgbm import LGBMClassifier
from threadpoolctl import threadpool_limits
from IPython.display import display,FileLink
threadpool_limits(2)
SEED=42
forbidden_attempts18=[]
def forbid_inference18(event,args):
 if event=='open' and args and isinstance(args[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(args[0])).name in {'test.csv','sample_submission.csv'}:
  forbidden_attempts18.append(str(args[0]));raise RuntimeError('Inference data access is forbidden in v18')
sys.addaudithook(forbid_inference18)
roots=[Path.cwd(),*Path.cwd().parents]
REPO=next((p for p in roots if (p/'data/train.csv').is_file()),None)
if REPO is None:raise FileNotFoundError('data/train.csv가 필요합니다.')
DATA=REPO/'data';WORK=REPO/'models'/'혜진' if (REPO/'models'/'혜진').is_dir() else REPO
OUT=WORK/'artifacts'/'v18';OUT.mkdir(parents=True,exist_ok=True)
# Inference files and submission directory are intentionally not used.
os.environ['MPLCONFIGDIR']=str(OUT/'.mplconfig')
train=pd.read_csv(DATA/'train.csv')
genes=[c for c in train if c not in ['ID','SUBCLASS']]
assert train.ID.is_unique
y=train.SUBCLASS.to_numpy();labels=np.sort(np.unique(y));yi=np.searchsorted(labels,y);NC=len(labels)
X=train[genes];groups=pd.util.hash_pandas_object(X,index=False).to_numpy()
dev,hold=next(StratifiedGroupKFold(5,shuffle=True,random_state=SEED).split(X,y,groups))
folds=list(StratifiedGroupKFold(3,shuffle=True,random_state=SEED+1).split(X.iloc[dev],y[dev],groups[dev]))
assert not set(groups[dev]).intersection(groups[hold])
for a,b in folds:assert not set(groups[dev[a]]).intersection(groups[dev[b]])
pd.DataFrame({'ID':train.ID,'split':np.where(np.isin(np.arange(len(train)),hold),'holdout','development'),'group':groups.astype(str)}).to_csv(OUT/'split_manifest.csv',index=False)
print('TRAIN ONLY',train.shape,'Dev/holdout',len(dev),len(hold),'OUT',OUT,flush=True)

from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier


TRAIN ONLY (6201, 4386) Dev/holdout 4960 1241 OUT /Users/mac/project/cancer_project/artifacts/v18


In [2]:
GROUPS = {
 'p53_cellcycle': 'TP53 RB1 CDKN2A CDKN2B CDK4 CDK6 CCND1 ATM CHEK2'.split(),
 'rtk_ras_mapk': 'EGFR ERBB2 ERBB3 KRAS NRAS HRAS BRAF NF1 MAP2K1 MAP2K2 KIT PDGFRA FGFR2 FGFR3'.split(),
 'pi3k_mtor': 'PIK3CA PIK3R1 PTEN AKT1 MTOR TSC1 TSC2 STK11'.split(),
 'dna_repair': 'BRCA1 BRCA2 PALB2 ATM ATR CHEK2 MLH1 MSH2 MSH6 PMS2 POLE POLD1'.split(),
 'chromatin': 'ARID1A ARID1B ARID2 PBRM1 SMARCA4 BAP1 SETD2 KMT2A KMT2C KMT2D ATRX'.split(),
 'wnt': 'APC CTNNB1 AXIN1 AXIN2 RNF43'.split(),
 'metabolism': 'IDH1 IDH2 SDHB SDHC SDHD FH VHL'.split(),
}

def kind(token):
 if 'fs' in token.lower(): return 'frameshift'
 if '*' in token: return 'stop'
 if 'del' in token.lower(): return 'deletion'
 if 'ins' in token.lower(): return 'insertion'
 m = re.fullmatch(r'([A-Z])(\d+)([A-Z])', token)
 if m: return 'synonymous' if m[1] == m[3] else 'substitution'
 return 'other'

class MutationFeatures(BaseEstimator, TransformerMixin):
 def __init__(self, mode='full', min_df=3):
  self.mode = mode
  self.min_df = min_df

 def dictionaries(self, X):
  genes = list(X.columns)
  for row in X.itertuples(index=False, name=None):
   d = {}; counts = {}; burden = 0; events = 0; positions = []
   changed = set()
   for gene, value in zip(genes, row):
    if value is None or str(value) in ('nan', '', '<NA>'):
     d['missing:' + gene] = 1; continue
    if value == 'WT': continue
    changed.add(gene); burden += 1; d['gene:' + gene] = 1
    if self.mode == 'binary': continue
    tokens = str(value).split()
    for raw_token in tokens:
     token = re.sub(r'^([A-Z]\d+)X$', r'\1*', raw_token)
     events += 1; typ = kind(token); counts[typ] = counts.get(typ, 0) + 1
     d['variant:' + gene + ':' + token] = 1
     d['type:' + gene + ':' + typ] = 1
     m = re.search(r'\d+', token)
     if m:
      position = int(m[0]); positions.append(position)
      d['position:' + gene + ':' + str(position)] = 1
   if self.mode == 'full':
    d['numeric:log_mutated_genes'] = np.log1p(burden)
    d['numeric:log_events'] = np.log1p(events)
    d['numeric:multi_event_excess'] = np.log1p(max(0, events-burden))
    for typ in ['frameshift','stop','deletion','insertion','synonymous','substitution','other']:
     d['numeric:log_' + typ] = np.log1p(counts.get(typ,0))
     d['numeric:fraction_' + typ] = counts.get(typ,0)/max(events,1)
    d['numeric:log_mean_position'] = np.log1p(np.mean(positions)) if positions else 0.
    for name, members in GROUPS.items():
     d['module:' + name] = np.log1p(len(changed.intersection(members)))
   yield d

 def fit(self, X, y=None):
  self.genes_ = list(X.columns)
  self.vectorizer_ = DictVectorizer(dtype=np.float32)
  matrix = self.vectorizer_.fit_transform(self.dictionaries(X))
  names = self.vectorizer_.get_feature_names_out()
  self.keep_ = (np.asarray((matrix != 0).sum(axis=0)).ravel() >= self.min_df) | np.array([s.startswith(('numeric:', 'module:')) for s in names])
  self.feature_names_ = names[self.keep_]
  return self

 def transform(self, X):
  missing = set(self.genes_) - set(X.columns)
  if missing: raise ValueError('Missing gene columns: ' + str(sorted(missing)[:10]))
  return sparse.csr_matrix(self.vectorizer_.transform(self.dictionaries(X[self.genes_]))[:, self.keep_])

 def get_feature_names_out(self, input_features=None): return self.feature_names_
def canon(t):
 t=t.strip().removeprefix('p.');t=re.sub(r'(?<=\d)X$','*',t)
 return re.sub(r'^([A-Z]\d+)(?:[A-Z*]+)?fs.*$',r'\1fs',t)
def aa_change(t):
 m=re.fullmatch(r'([A-Z])\d+([A-Z*])',t)
 if m:return m[1]+'>'+m[2]
 m=re.match(r'([A-Z])\d+',t)
 if m:
  for suffix in ['fs','del','ins']:
   if suffix in t:return m[1]+'>'+suffix
 return None

def parse_events(frame):
 names=list(frame.columns);result=[]
 for row in frame.itertuples(index=False,name=None):
  result.append([(g,tuple(sorted(set(canon(t) for t in v.split())))) for g,v in zip(names,row) if isinstance(v,str) and v!='WT'])
 return result

class EvidenceEncoder:
 def __init__(self,unknown=1.,silent=1.,support=2):self.unknown=unknown;self.silent=silent;self.support=support
 def fit(self,events):
  self.token_df={};self.change_df={}
  if self.unknown<1:
   tokens=Counter();changes=Counter()
   for row in events:
    for g,ts in row:
     tokens.update((g,t) for t in ts)
     changes.update(set((g,a) for t in ts if (a:=aa_change(t)) is not None))
   self.token_df={k:v for k,v in tokens.items() if v>=self.support}
   self.change_df={k:v for k,v in changes.items() if v>=self.support}
  return self
 def transform(self,events,training=False,exclude_events=None):
  result=[]
  for row_number,row in enumerate(events):
   own=exclude_events[row_number] if exclude_events is not None else (row if training else [])
   own_tokens={(gg,tt) for gg,tts in own for tt in tts}
   own_changes={(gg,ac) for gg,tts in own for tt in tts if (ac:=aa_change(tt)) is not None}
   d={};gene_weights={};type_sums=Counter();non_syn=0.;trunc=0.;multikind=0
   for g,ts in row:
    types={};gene=0.;known_gene=False
    for t in ts:
     typ=kind(t);a=aa_change(t)
     known=self.token_df.get((g,t),0)-int((g,t) in own_tokens)>=self.support
     known_aa=a is not None and self.change_df.get((g,a),0)-int((g,a) in own_changes)>=self.support
     if typ=='synonymous':w=self.silent
     elif known or typ in ['stop','frameshift','deletion','insertion']:w=1.
     elif known_aa:w=max(self.unknown,.75)
     else:w=self.unknown
     if w<=0:continue
     gene=max(gene,w);types[typ]=max(types.get(typ,0),w)
     d['variant:'+g+':'+t]=max(d.get('variant:'+g+':'+t,0),w)
     m=re.search(r'\d+',t)
     if m:d['position:'+g+':'+m[0]]=max(d.get('position:'+g+':'+m[0],0),w)
     if a:d['aa:'+g+':'+a]=max(d.get('aa:'+g+':'+a,0),w)
     if known and typ!='synonymous':known_gene=True
    if gene<=0:continue
    d['gene:'+g]=gene;gene_weights[g]=gene
    for typ,w in types.items():d['type:'+g+':'+typ]=w;type_sums[typ]+=w
    if known_gene:d['supported:'+g]=1.
    non_syn+=max([w for k,w in types.items() if k!='synonymous'] or [0.])
    trunc+=max([w for k,w in types.items() if k in ['stop','frameshift','deletion','insertion']] or [0.])
    multikind+=len(types)>1
   n=len(row)
   d['numeric:log_raw_genes']=np.log1p(n)
   d['numeric:log_effective_genes']=np.log1p(sum(gene_weights.values()))
   d['numeric:log_nonsyn_genes']=np.log1p(non_syn)
   d['numeric:log_trunc_genes']=np.log1p(trunc)
   d['numeric:log_multikind_genes']=np.log1p(multikind)
   for typ in ['synonymous','substitution','stop','frameshift','deletion','insertion','other']:
    d['numeric:gene_fraction_'+typ]=type_sums[typ]/max(n,1)
   for name,gs in GROUPS.items():d['module:'+name]=np.log1p(sum(gene_weights.get(g,0) for g in gs))
   result.append(d)
  return result

events=parse_events(X)
legacy_records=list(MutationFeatures(mode='full').dictionaries(X))
assert canon('R213X')=='R213*' and canon('K16Rfs')=='K16fs'
assert aa_change('R132H')==aa_change('R140H')=='R>H'
print('Parsed rows:',len(events),'Normalization examples passed')


Parsed rows: 6201 Normalization examples passed


In [3]:
def relevance_scores(a,target,method):
 base=method.removesuffix('_full')
 if base=='embedded':
  scaled=csr32(MaxAbsScaler().fit_transform(a))
  fitted=OneVsRestClassifier(LogisticRegression(solver='liblinear',l1_ratio=1,C=.2,class_weight='balanced',max_iter=2000),n_jobs=1).fit(scaled,target)
  return np.max(np.abs(np.vstack([e.coef_[0] for e in fitted.estimators_])),axis=0)
 return (chi2(a,target) if base=='chi2' else f_classif(a,target))[0]

def csr32(a):
 b=sparse.csr_matrix(a,dtype=np.float32)
 if b.nnz>=np.iinfo(np.int32).max:raise ValueError('Sparse matrix too large for tree index type')
 b.indices=b.indices.astype(np.int32,copy=False);b.indptr=b.indptr.astype(np.int32,copy=False)
 return b

class Space:
 def fit(self,rs):
  self.vec=DictVectorizer(dtype=np.float32);a=self.vec.fit_transform(rs)
  names=self.vec.get_feature_names_out();df=np.asarray((a!=0).sum(axis=0)).ravel()
  keep=(df>=3)|np.array([n.startswith(('numeric:','module:')) for n in names])
  self.raw=np.flatnonzero(keep);self.names=names[keep]
  return csr32(a[:,self.raw])
 def transform(self,rs):return csr32(self.vec.transform(rs)[:,self.raw])
 def compact_vec(self,indices):
  v=copy.deepcopy(self.vec);mask=np.zeros(len(v.feature_names_),bool);mask[self.raw[np.sort(indices)]]=True
  v.restrict(mask);return v

class Selector:
 def __init__(self,a,target,names):
  self.a=a;self.y=target;self.names=names;self.cache={};self.corr={};self.scores={};self.audit=[];self.examples=[]
 def order(self,method,threshold):
  key=(method,threshold)
  if key in self.cache:return self.cache[key]
  if method not in self.scores:
   with warnings.catch_warnings():
    warnings.simplefilter('ignore',RuntimeWarning);score=relevance_scores(self.a,self.y,method)
   score=np.nan_to_num(score,nan=0,posinf=np.finfo(np.float32).max,neginf=0)
   if not method.endswith('_full'):score[np.array([n.startswith('numeric:') for n in self.names])]=0
   self.scores[method]=score
  score=self.scores[method]
  if method not in self.corr:
   pool=np.argsort(-score,kind='stable')[:min(4096,int((score>0).sum()))]
   z=self.a[:,pool];mean=np.asarray(z.mean(axis=0)).ravel();var=np.asarray(z.multiply(z).mean(axis=0)).ravel()-mean**2
   sd=np.sqrt(np.maximum(var,1e-12));corr=(z.T@z).toarray()/z.shape[0]-np.outer(mean,mean)
   corr/=np.outer(sd,sd);np.clip(corr,-1,1,out=corr)
   self.corr[method]=(pool,corr)
  pool,corr=self.corr[method];chosen=[]
  for i in range(len(pool)):
   if chosen:
    correlations=np.abs(corr[i,chosen]);j=int(np.argmax(correlations))
    if correlations[j]>=threshold:
     if len(self.examples)<100:self.examples.append({'method':method,'threshold':threshold,'kept':self.names[pool[chosen[j]]],'discarded':self.names[pool[i]],'correlation':float(corr[i,chosen[j]])})
     continue
   chosen.append(i)
   if len(chosen)>=2048:break
  order=pool[chosen];self.cache[key]=order
  self.audit.append({'method':method,'threshold':threshold,'association_pool':len(pool),'remaining_up_to_2048':len(order)})
  return order
 def select(self,method,threshold,k):return np.sort(self.order(method,threshold)[:k])

class CompactPredictor:
 def __init__(self,space,cols,est):
  self.vec=space.compact_vec(cols);self.names=self.vec.get_feature_names_out();self.est=est
  self.direct_inputs=len(self.names);self.raw_inputs=len(self.names);self.kind='compact'
 def predict(self,rs):return self.est.predict_proba(csr32(self.vec.transform(rs)))

def background_frame(frame,training_frame,seed,count):
 rng=np.random.default_rng(seed);out=frame.copy();arr=out.to_numpy(copy=True)
 rate=(training_frame.notna()&training_frame.ne('WT')).mean().to_numpy();prob=rate/rate.sum();idx=np.flatnonzero(prob>0)
 for r in range(len(arr)):
  n=min(len(idx),int(rng.poisson(count)))
  for c in rng.choice(len(prob),size=n,replace=False,p=prob):
   token='A1A' if rng.random()<.5 else 'A1V'
   arr[r,c]=token if not isinstance(arr[r,c],str) or arr[r,c]=='WT' else arr[r,c]+' '+token
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)
class ColumnPost:
 def __init__(self,mode='none'):self.mode=mode
 def fit(self,a,names):
  self.raw_names=np.array(names);self.names=np.array(names)
  if self.mode=='maxabs':self.scaler=MaxAbsScaler().fit(a)
  if self.mode=='block':
   self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names])
   self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])
   self.tf=TfidfTransformer(norm='l2',use_idf=True).fit(a[:,self.binary])
   self.scaler=MaxAbsScaler().fit(a[:,self.summary])
   self.names=np.concatenate([self.raw_names[self.binary],self.raw_names[self.summary]])
  return self
 def transform(self,a):
  if self.mode=='maxabs':return csr32(self.scaler.transform(a))
  if self.mode=='block':return csr32(sparse.hstack([self.tf.transform(a[:,self.binary]),self.scaler.transform(a[:,self.summary])],format='csr'))
  return csr32(a)

class Member:
 def __init__(self,name,view,encoder,space,cols,post,est):
  self.name=name;self.view=view;self.encoder=encoder;self.vec=space.compact_vec(cols);self.post=post;self.est=est
  self.raw_names=self.vec.get_feature_names_out();self.names=post.names
 def predict_rows(self,rs):return self.est.predict_proba(self.post.transform(csr32(self.vec.transform(rs))))

VIEWS={'raw':(1.,1.),'syn_down':(1.,.25),'gated':(.25,1.),'gated_syn_down':(.25,.25),'gated_nonsyn':(.25,0.)}
CONFIGS={}
for view in VIEWS:
 for k in [1024,2048]:
  for c in [.1,.5]:CONFIGS[f'{view}_lr_k{k}_C{c}']=dict(view=view,algo='lr',k=k,C=c,post='none',method='chi2_full')
  CONFIGS[f'{view}_lgb_k{k}']=dict(view=view,algo='lgb',k=k,post='none',method='f_classif_full')
for view in ['raw','gated_syn_down']:
 for depth in [3,5]:CONFIGS[f'{view}_xgb_d{depth}']=dict(view=view,algo='xgb',k=2048,depth=depth,post='none',method='f_classif_full')
 for alpha in [1,10]:CONFIGS[f'{view}_mlp_a{alpha}']=dict(view=view,algo='mlp',k=1024,alpha=alpha,post='maxabs',method='chi2_full')
 for c in [1,4]:CONFIGS[f'{view}_blocklr_C{c}']=dict(view=view,algo='lr',k=2048,C=c,post='block',method='chi2_full')
 CONFIGS[f'{view}_extra']=dict(view=view,algo='extra',k=1024,post='none',method='f_classif_full')
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))

def estimator(cfg):
 algo=cfg['algo']
 if algo=='lr':return LogisticRegression(C=cfg['C'],class_weight='balanced',max_iter=2000)
 if algo=='lgb':return LGBMClassifier(n_estimators=300,learning_rate=.04,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
 if algo=='xgb':return XGBClassifier(n_estimators=300,max_depth=cfg['depth'],learning_rate=.04,min_child_weight=3,reg_lambda=10,subsample=.85,colsample_bytree=.85,max_bin=63,tree_method='hist',objective='multi:softprob',eval_metric='mlogloss',n_jobs=2,random_state=SEED)
 if algo=='mlp':return MLPClassifier(hidden_layer_sizes=(128,),activation='relu',alpha=cfg['alpha'],learning_rate_init=.001,batch_size=128,max_iter=250,tol=1e-4,n_iter_no_change=15,early_stopping=False,random_state=SEED)
 return ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)

def prepare_view(idx,view):
 ev=[events[i] for i in idx];enc=EvidenceEncoder(*VIEWS[view]).fit(ev)
 rs=enc.transform(ev,training=True);space=Space();a=space.fit(rs);selector=Selector(a,yi[idx],space.names)
 return enc,space,a,selector

def train_new(name,idx,cache):
 cfg=CONFIGS[name];view=cfg['view']
 if view not in cache:cache[view]=prepare_view(idx,view)
 enc,space,a,selector=cache[view];cols=selector.select(cfg['method'],.98,cfg['k'])
 post=ColumnPost(cfg['post']).fit(a[:,cols],space.names[cols]);tx=post.transform(a[:,cols]);target=yi[idx]
 row_weights=np.ones(len(idx),dtype=float)
 if cfg.get('augment',False):
  if '_aug_events' not in cache:
   cache['_aug_events']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))
  aug_rs=enc.transform(cache['_aug_events'],exclude_events=[events[i] for i in idx])
  aug_x=post.transform(space.transform(aug_rs)[:,cols]);tx=csr32(sparse.vstack([tx,aug_x],format='csr'))
  target=np.concatenate([target,target]);row_weights=np.concatenate([row_weights,.5*row_weights])
 est=estimator(cfg)
 if cfg['algo'] in ['xgb','mlp']:est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*row_weights)
 elif cfg.get('augment',False):est.fit(tx,target,sample_weight=row_weights)
 else:est.fit(tx,target)
 return Member(name,view,enc,space,cols,post,est)

LEGACY={
 'legacy_lr':dict(k=2048,method='chi2',algo='lr'),
 'legacy_lgb':dict(k=2048,method='f_classif_full',algo='lgb'),
 'legacy_extra':dict(k=1024,method='f_classif_full',algo='extra')}
def train_legacy(idx):
 space=Space();a=space.fit([legacy_records[i] for i in idx]);sel=Selector(a,yi[idx],space.names);result=[]
 for name,cfg in LEGACY.items():
  cols=sel.select(cfg['method'],.95,cfg['k'])
  if cfg['algo']=='lr':est=LogisticRegression(C=.1,class_weight='balanced',max_iter=2000)
  elif cfg['algo']=='lgb':est=LGBMClassifier(n_estimators=250,learning_rate=.05,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
  else:est=ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)
  est.fit(a[:,cols],yi[idx]);post=ColumnPost().fit(a[:,cols],space.names[cols]);result.append(Member(name,'legacy',None,space,cols,post,est))
 return result

def duplicate_frame(frame):
 arr=frame.to_numpy(copy=True)
 for i,row in enumerate(arr):
  for j,v in enumerate(row):
   if isinstance(v,str) and v!='WT':arr[i,j]=v+' '+v
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)

def rare_background(frame,tr_events,seed,count=20):
 rng=np.random.default_rng(seed);df=Counter((g,t) for row in tr_events for g,ts in row for t in ts)
 syn=[];sub=[]
 for (g,t),n in df.items():
  if n>2:continue
  if kind(t)=='synonymous':syn.append((g,t))
  elif kind(t)=='substitution':sub.append((g,t))
 arr=frame.to_numpy(copy=True);lookup={g:i for i,g in enumerate(frame.columns)}
 for row in arr:
  for _ in range(int(rng.poisson(count))):
   pool=syn if rng.random()<.5 else sub
   g,t=pool[int(rng.integers(len(pool)))];j=lookup[g]
   row[j]=t if not isinstance(row[j],str) or row[j]=='WT' else row[j]+' '+t
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)

def eval_views(idx,tr_idx,seed):
 clean=X.iloc[idx];noise=rare_background(clean,[events[i] for i in tr_idx],seed);dup=duplicate_frame(clean)
 frames=[clean,noise,dup]
 return frames,[parse_events(f) for f in frames],[list(MutationFeatures(mode='full').dictionaries(f)) for f in frames]

def predict_members(members,weights,frame):
 ev=None;old=None;cache={};p=None
 for m,w in zip(members,weights):
  if m.view=='legacy':
   if old is None:old=list(MutationFeatures(mode='full').dictionaries(frame[genes]))
   rs=old
  else:
   if ev is None:ev=parse_events(frame[genes])
   if m.view not in cache:cache[m.view]=m.encoder.transform(ev)
   rs=cache[m.view]
  q=m.predict_rows(rs);p=q*w if p is None else p+q*w
 return p
print('Novel settings:',len(CONFIGS),'Views:',VIEWS)

# CV-guided targeted refinement: regularization, support threshold, and rare-variant augmentation.
VIEWS['gated_syn_down_s3']=(.25,.25,3)
CONFIGS.update({'raw_mlp_a0.01': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'raw_mlp_a0.1': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.01': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.1': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k1024': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 1024, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_lr_k2048_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k2048_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k2048': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d3': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 3, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d5': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full'}, 'raw_aug_lgb': {'view': 'raw', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'raw_aug_xgb_d5': {'view': 'raw', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'gated_s3_aug_lgb': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}})
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))
print('Total new configurations after refinement:',len(CONFIGS))


Novel settings: 44 Views: {'raw': (1.0, 1.0), 'syn_down': (1.0, 0.25), 'gated': (0.25, 1.0), 'gated_syn_down': (0.25, 0.25), 'gated_nonsyn': (0.25, 0.0)}
Total new configurations after refinement: 59


In [4]:
# Freeze the previous version; do not repeat its 59-setting search.
BASE_CONFIGS=copy.deepcopy(CONFIGS)
BASE_PLAN=[('v4_reference',.25),('gated_s3_aug_lgb',.1875),('raw_aug_xgb_d5',.1875),('raw_lr_k2048_C0.1',.1875),('gated_syn_down_mlp_a1',.1875)]

def fit_base(idx):
 cache={};ms=[];ws=[]
 for name,w in BASE_PLAN:
  if name=='v4_reference':ms.extend(train_legacy(idx));ws.extend([w/3]*3)
  else:ms.append(train_new(name,idx,cache));ws.append(w)
 return ms,np.array(ws)


In [5]:
class NewSelector:
 def __init__(self,a,target,names):self.a=a;self.y=target;self.names=names;self.cache={}
 def select(self,method,k):
  if (method,k) in self.cache:return self.cache[(method,k)]
  numeric=np.array([s.startswith(('numeric:','module:')) for s in self.names])
  with warnings.catch_warnings():
   warnings.simplefilter('ignore',RuntimeWarning)
   global_score=np.nan_to_num(f_classif(self.a,self.y)[0],nan=0,posinf=0,neginf=0)
   if method=='balanced':
    orders=[]
    for c in range(NC):
     yes=self.y==c
     score=np.nan_to_num(chi2(self.a,yes)[0],nan=0,posinf=0,neginf=0)
     high=np.asarray(self.a[yes].mean(axis=0)).ravel()>np.asarray(self.a[~yes].mean(axis=0)).ravel()
     score[~high|numeric]=0
     orders.append(np.argsort(-score,kind='stable')[:min(k,int((score>0).sum()))])
    order=[]
    for rank in range(max(map(len,orders))):
     for o in orders:
      if rank<len(o):order.append(int(o[rank]))
    order+=list(np.argsort(-global_score,kind='stable'))
   else:order=list(np.argsort(-global_score,kind='stable'))
  z=self.a.tocsc();chosen=list(np.flatnonzero(numeric));seen=set(chosen);signatures=set()
  for j in order:
   j=int(j)
   if j in seen:continue
   seen.add(j)
   if global_score[j]<=0:continue
   lo,hi=z.indptr[j:j+2]
   sig=hashlib.sha256(z.indices[lo:hi].tobytes()+z.data[lo:hi].tobytes()).digest()
   if sig in signatures:continue
   signatures.add(sig);chosen.append(j)
   if len(chosen)>=k:break
  self.cache[(method,k)]=np.sort(chosen);return self.cache[(method,k)]

class SpecificityPost:
 def fit(self,a,names,target):
  self.names=np.array(names);self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names]);self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])
  z=(a[:,self.binary]!=0).astype(np.float32)
  counts=np.vstack([np.asarray(z[target==c].sum(axis=0)).ravel() for c in range(NC)])
  prior=np.bincount(target,minlength=NC)/len(target);freq=counts.sum(axis=0)
  posterior=(counts+10*prior[:,None])/(freq+10)
  kl=(posterior*np.log(np.maximum(posterior,1e-12)/prior[:,None])).sum(axis=0)
  self.weights=(.25+np.sqrt(np.maximum(kl,0))*freq/(freq+20)).astype(np.float32)
  self.scaler=MaxAbsScaler().fit(a[:,self.summary]);return self
 def transform(self,a):
  from sklearn.preprocessing import normalize
  block=normalize(a[:,self.binary].multiply(self.weights),norm='l2')
  return csr32(sparse.hstack([block,self.scaler.transform(a[:,self.summary])],format='csr'))

NEW={}
for view in ['raw','gated_syn_down_s3']:
 for method in ['global','balanced']:
  NEW[f'{view}_{method}_lgb4096']=dict(view=view,algo='lgb',k=4096,method=method)
for c in [.1,.5]:NEW[f'balanced_lr_C{c}']=dict(view='raw',algo='lr',k=2048,method='balanced',C=c)
for c in [1.,4.]:NEW[f'specificity_lr_C{c}']=dict(view='raw',algo='lr',k=4096,method='balanced',C=c,specificity=True)

def fit_new6(name,idx,cache):
 cfg=NEW[name];view=cfg['view']
 if view not in cache:
  enc,space,a,_=prepare_view(idx,view);cache[view]=(enc,space,a,NewSelector(a,yi[idx],space.names))
 enc,space,a,sel=cache[view];cols=sel.select(cfg['method'],cfg['k']);target=yi[idx]
 if cfg.get('specificity'):post=SpecificityPost().fit(a[:,cols],space.names[cols],target)
 else:post=ColumnPost().fit(a[:,cols],space.names[cols])
 est=estimator(cfg);tx=post.transform(a[:,cols])
 if cfg.get('augment'):
  if '_aug' not in cache:cache['_aug']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))
  ar=enc.transform(cache['_aug'],exclude_events=[events[i] for i in idx]);ax=post.transform(space.transform(ar)[:,cols])
  tx=csr32(sparse.vstack([tx,ax],format='csr'));target=np.concatenate([target,target]);rw=np.r_[np.ones(len(idx)),np.full(len(idx),.5)]
 else:rw=np.ones(len(idx))
 if cfg['algo']=='xgb':est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*rw)
 elif cfg.get('augment'):est.fit(tx,target,sample_weight=rw)
 else:est.fit(tx,target)
 m=Member(name,view,enc,space,cols,post,est)
 # Column order of a specificity block changes, but raw_names always describe the vectorizer input.
 if cfg.get('specificity'):m.names=np.concatenate([m.raw_names[post.binary],m.raw_names[post.summary]])
 return m
(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))
print('New configurations',len(NEW))

NEW.update({'gated_s3_aug_global_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'global', 'augment': True}, 'gated_s3_aug_balanced_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'balanced', 'augment': True}, 'raw_aug_global_xgb4096_d3': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 3, 'augment': True}, 'raw_aug_global_xgb4096_d5': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 5, 'augment': True}})
(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))

print('Total new configurations including augmentation extension:',len(NEW))


New configurations 8
Total new configurations including augmentation extension: 12


In [6]:
def fit_plan6(idx,plan,basefit=None):
 ms=[];ws=[];cache={}
 for name,w in plan:
  if name=='v5_reference':
   old,ow=basefit if basefit is not None else fit_base(idx)
   ms.extend(old);ws.extend(ow*w)
  else:ms.append(fit_new6(name,idx,cache));ws.append(w)
 return ms,np.array(ws)

V6_PLAN=[('v5_reference',.75),('gated_s3_aug_global_lgb4096',.25)]
def fit_v6(idx):return fit_plan6(idx,V6_PLAN)


In [7]:
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.feature_selection import SelectKBest

def expand10(raw):
 t=canon(raw);m=re.fullmatch(r'([0-9]+)_([0-9]+)([A-Z]+)>([A-Z]+)',t)
 if m and int(m[2])-int(m[1])+1==len(m[3])==len(m[4]):
  return tuple(f'{a}{int(m[1])+j}{b}' for j,(a,b) in enumerate(zip(m[3],m[4])))
 return (t,)

def kind10(t):
 if 'fs' in t:return 'frameshift'
 if t.endswith('*'):return 'stop'
 if 'delins' in t:return 'delins'
 if 'del' in t:return 'deletion'
 if 'ins' in t:return 'insertion'
 m=re.fullmatch(r'([A-Z])[0-9]+([A-Z])',t)
 if m:return 'synonymous' if m[1]==m[2] else 'substitution'
 return 'other'

def packet10(frame,ev=None):
 if ev is None:ev=parse_events(frame[genes])
 resolved=[[(g,tuple(sorted({s for t in ts for s in expand10(t)}))) for g,ts in row] for row in ev]
 missing=frame[genes].isna().to_numpy()
 return dict(events=resolved,missing=missing)

def records10(packet,representation,numeric=True):
 rs=[]
 for i,row in enumerate(packet['events']):
  d={};syn=nonsyn=structural=unparsed=0
  for g,ts in row:
   d['gene:'+g]=1.;types={kind10(t) for t in ts}
   sy='synonymous' in types;ns=bool(types.intersection({'substitution','stop','frameshift','deletion','insertion','delins'}));st=bool(types.intersection({'stop','frameshift','deletion','insertion','delins'}));ot='other' in types
   syn+=sy;nonsyn+=ns;structural+=st;unparsed+=ot
   if representation=='type':
    for typ in types:d['type:'+g+':'+typ]=1.
   elif representation=='functional':
    for flag,family in [(sy,'syn_gene'),(ns,'nonsyn_gene'),(st,'structural_gene'),(ot,'unparsed_gene')]:
     if flag:d[family+':'+g]=1.
  if numeric:
   d['numeric:log_observed_mutated_genes']=np.log1p(len(row));d['numeric:log_missing_genes']=np.log1p(int(packet['missing'][i].sum()))
   if representation!='gene':
    for title,value in [('syn',syn),('nonsyn',nonsyn),('structural',structural),('unparsed',unparsed)]:
     d['numeric:log_'+title+'_genes']=np.log1p(value);d['numeric:fraction_'+title+'_genes']=value/max(len(row),1)
  rs.append(d)
 return rs

def mask_packet10(packet,probability,seed):
 rng=np.random.default_rng(seed);out=[];missing=packet['missing'].copy();gpos={g:j for j,g in enumerate(genes)}
 for i,row in enumerate(packet['events']):
  keep=[]
  for g,ts in row:
   if rng.random()<probability:missing[i,gpos[g]]=True
   else:keep.append((g,ts))
  out.append(keep)
 return dict(events=out,missing=missing)

def select_cols10(a,target,k):
 if k is None or a.shape[1]<=k:return np.arange(a.shape[1])
 with warnings.catch_warnings():
  warnings.simplefilter('ignore',RuntimeWarning);scores=np.nan_to_num(chi2(a,target)[0],nan=0,posinf=0,neginf=0)
 return np.sort(np.argsort(-scores,kind='stable')[:k])

class MissingBernoulli10:
 def __init__(self,alpha=.1,pooled=False,fit_prior=False,k=None):self.alpha=alpha;self.pooled=pooled;self.fit_prior=fit_prior;self.k=k
 def fit(self,a,target,missing,sample_weight=None):
  self.cols=select_cols10(a,target,self.k);a=csr32(a[:,self.cols]);missing=csr32(missing[:,self.cols]);w=np.ones(len(target)) if sample_weight is None else np.asarray(sample_weight)
  self.classes_=np.unique(target);nclass=np.array([w[target==c].sum() for c in self.classes_])
  counts=np.vstack([np.asarray(a[target==c].multiply(w[target==c,None]).sum(axis=0)).ravel() for c in self.classes_])
  absent=np.vstack([np.asarray(missing[target==c].multiply(w[target==c,None]).sum(axis=0)).ravel() for c in self.classes_])
  observed=nclass[:,None]-absent
  if self.pooled:
   denominator=w.sum()-np.asarray(missing.multiply(w[:,None]).sum(axis=0)).ravel()
   prior=np.clip(np.asarray(a.multiply(w[:,None]).sum(axis=0)).ravel()/np.maximum(denominator,1e-9),1e-7,1-1e-7)
   prob=(counts+self.alpha*prior)/(observed+self.alpha)
  else:prob=(counts+self.alpha)/(observed+2*self.alpha)
  prob=np.clip(prob,1e-7,1-1e-7);self.log0=np.log1p(-prob);self.logit=np.log(prob)-self.log0
  prior=nclass/nclass.sum() if self.fit_prior else np.full(len(nclass),1/len(nclass));self.logprior=np.log(prior)
  self.coef_=self.logit;return self
 def predict_proba(self,a,missing):
  a=csr32(a[:,self.cols]);missing=csr32(missing[:,self.cols])
  score=np.asarray(a@self.logit.T)+self.log0.sum(axis=1)+self.logprior-np.asarray(missing@self.log0.T)
  return softmax(score,axis=1)

class GroupSigmoidSVM10:
 def __init__(self,C=.1,k=2048,balanced_calibration=False):self.C=C;self.k=k;self.balanced_calibration=balanced_calibration
 def fit(self,a,target,profile_groups,sample_weight):
  self.classes_=np.unique(target);oof=np.zeros((len(target),len(self.classes_)),float)
  for tr,va in StratifiedGroupKFold(3,shuffle=True,random_state=310).split(a,target,profile_groups):
   assert not set(profile_groups[tr]).intersection(profile_groups[va])
   cols=select_cols10(a[tr],target[tr],self.k);scaler=MaxAbsScaler().fit(a[tr][:,cols])
   est=LinearSVC(C=self.C,class_weight='balanced',dual='auto',max_iter=5000,random_state=42).fit(csr32(scaler.transform(a[tr][:,cols])),target[tr],sample_weight=sample_weight[tr])
   scores=est.decision_function(csr32(scaler.transform(a[va][:,cols])))
   assert np.array_equal(est.classes_,self.classes_);oof[va]=scores
  self.calibrators=[]
  for j,c in enumerate(self.classes_):
   cal=LogisticRegression(C=1.,class_weight='balanced' if self.balanced_calibration else None,max_iter=1000).fit(oof[:,j,None],(target==c).astype(int),sample_weight=sample_weight);self.calibrators.append(cal)
  self.cols=select_cols10(a,target,self.k);self.scaler=MaxAbsScaler().fit(a[:,self.cols]);self.est=LinearSVC(C=self.C,class_weight='balanced',dual='auto',max_iter=5000,random_state=42).fit(csr32(self.scaler.transform(a[:,self.cols])),target,sample_weight=sample_weight);self.coef_=self.est.coef_;return self
 def predict_proba(self,a):
  score=self.est.decision_function(csr32(self.scaler.transform(a[:,self.cols])))
  p=np.column_stack([cal.predict_proba(score[:,j,None])[:,1] for j,cal in enumerate(self.calibrators)])
  return p/np.maximum(p.sum(axis=1,keepdims=True),1e-12)

class Member10:
 def __init__(self,name,space,est,representation,numeric,cols=None,scaler=None):
  self.name=name;self.view='v10_'+representation;self.representation=representation;self.numeric=numeric;self.vec=space.compact_vec(np.arange(len(space.names)));self.est=est;self.cols=est.cols if cols is None else cols;self.raw_names=space.names[self.cols];self.names=self.raw_names;self.scaler=scaler
  lookup={g:i for i,g in enumerate(genes)};self.feature_gene_positions=np.array([lookup.get(str(s).split(':')[1],-1) if ':' in s and not str(s).startswith('numeric:') else -1 for s in space.names])
 def missing_matrix(self,packet):
  out=np.zeros((len(packet['events']),len(self.feature_gene_positions)),bool);valid=self.feature_gene_positions>=0
  out[:,valid]=packet['missing'][:,self.feature_gene_positions[valid]]
  return csr32(sparse.csr_matrix(out))
 def predict_packet(self,packet,records=None):
  rs=records10(packet,self.representation,self.numeric) if records is None else records;a=csr32(self.vec.transform(rs))
  if isinstance(self.est,MissingBernoulli10):return self.est.predict_proba(a,self.missing_matrix(packet))
  if isinstance(self.est,GroupSigmoidSVM10):return self.est.predict_proba(a)
  tx=a[:,self.cols]
  if self.scaler is not None:tx=csr32(self.scaler.transform(tx))
  return self.est.predict_proba(tx)
 def predict_frame(self,frame):return self.predict_packet(packet10(frame[genes]))

predict_members6=predict_members
def predict_members(members,weights,frame):
 old=[(m,w) for m,w in zip(members,weights) if not hasattr(m,'predict_frame')];p=predict_members6([m for m,w in old],[w for m,w in old],frame) if old else None
 packet=None
 for m,w in zip(members,weights):
  if hasattr(m,'predict_frame'):
   if packet is None:packet=packet10(frame[genes])
   q=m.predict_packet(packet)*w;p=q if p is None else p+q
 return p

C10={
 'gene_nb_a0.1':dict(rep='gene',algo='nb',alpha=.1,pooled=False,fit_prior=False,k=None,mask=0.),
 'gene_nb_a1':dict(rep='gene',algo='nb',alpha=1.,pooled=False,fit_prior=False,k=None,mask=0.),
 'gene_nb_empirical_prior':dict(rep='gene',algo='nb',alpha=.1,pooled=False,fit_prior=True,k=None,mask=0.),
 'gene_nb_pooled20':dict(rep='gene',algo='nb',alpha=20.,pooled=True,fit_prior=False,k=None,mask=0.),
 'type_nb_pooled20':dict(rep='type',algo='nb',alpha=20.,pooled=True,fit_prior=False,k=4096,mask=0.),
 'gene_lr':dict(rep='gene',algo='lr',C=.1,k=2048,mask=0.),
 'gene_lr_mask10':dict(rep='gene',algo='lr',C=.1,k=2048,mask=.1),
 'gene_lr_mask25':dict(rep='gene',algo='lr',C=.1,k=2048,mask=.25),
 'type_lr':dict(rep='type',algo='lr',C=.1,k=2048,mask=0.),
 'type_lr_mask10':dict(rep='type',algo='lr',C=.1,k=2048,mask=.1),
 'functional_lr':dict(rep='functional',algo='lr',C=.1,k=2048,mask=0.),
 'functional_lr_mask10':dict(rep='functional',algo='lr',C=.1,k=2048,mask=.1),
 'gene_svm':dict(rep='gene',algo='svm',C=.1,k=2048,mask=0.),
 'type_svm':dict(rep='type',algo='svm',C=.1,k=2048,mask=0.),
 'functional_svm':dict(rep='functional',algo='svm',C=.1,k=2048,mask=0.),
 'functional_svm_mask10':dict(rep='functional',algo='svm',C=.1,k=2048,mask=.1),
 'functional_lgb':dict(rep='functional',algo='lgb',k=2048,mask=0.),
 'functional_lgb_mask10':dict(rep='functional',algo='lgb',k=2048,mask=.1),
 'functional_lgb_mask25':dict(rep='functional',algo='lgb',k=2048,mask=.25)}

def prepare10(idx,representation,numeric,cache):
 key=(representation,numeric)
 if key not in cache:
  if '_clean' not in cache:cache['_clean']=packet10(X.iloc[idx],[events[i] for i in idx])
  rs=records10(cache['_clean'],representation,numeric);space=Space();a=space.fit(rs);cache[key]=(space,a)
 return cache[key]

def train10(name,idx,cache):
 cfg=C10[name];numeric=cfg['algo'] not in ['nb','cnb'];space,a=prepare10(idx,cfg['rep'],numeric,cache);target=yi[idx];rw=np.ones(len(idx));train_groups=groups[idx]
 augmented=None
 if cfg['mask']:
  key=('masked',cfg['mask'])
  if key not in cache:cache[key]=mask_packet10(cache['_clean'],cfg['mask'],1042+int(cfg['mask']*100))
  augmented=cache[key];ar=records10(augmented,cfg['rep'],numeric);a_aug=space.transform(ar);a=csr32(sparse.vstack([a,a_aug],format='csr'));target=np.r_[target,target];train_groups=np.r_[train_groups,train_groups];rw=np.r_[rw,.5*rw]/1.5
 if cfg['algo']=='nb':
  est=MissingBernoulli10(cfg['alpha'],cfg['pooled'],cfg['fit_prior'],cfg['k'])
  temp=Member10(name,space,est,cfg['rep'],False,cols=np.arange(a.shape[1]));missing=temp.missing_matrix(cache['_clean']);est.fit(a,target,missing,sample_weight=rw)
  return Member10(name,space,est,cfg['rep'],False)
 if cfg['algo']=='cnb':
  cols=select_cols10(a,target,cfg['k']);est=ComplementNB(alpha=cfg['alpha'],norm=False).fit(a[:,cols],target,sample_weight=rw)
  return Member10(name,space,est,cfg['rep'],False,cols=cols)
 if cfg['algo']=='svm':
  est=GroupSigmoidSVM10(cfg['C'],cfg['k'],cfg.get('balanced_calibration',False)).fit(a,target,train_groups,rw);return Member10(name,space,est,cfg['rep'],True)
 # Outer selection and scaler are fit on the original training patients only.
 cols=select_cols10(prepare10(idx,cfg['rep'],numeric,cache)[1],yi[idx],cfg['k'])
 scaler=MaxAbsScaler().fit(prepare10(idx,cfg['rep'],numeric,cache)[1][:,cols]) if cfg['algo']=='lr' else None
 tx=a[:,cols];tx=csr32(scaler.transform(tx)) if scaler is not None else csr32(tx)
 est=LogisticRegression(C=cfg['C'],class_weight='balanced',max_iter=2000) if cfg['algo']=='lr' else LGBMClassifier(n_estimators=300,learning_rate=.04,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=42)
 est.fit(tx,target,sample_weight=rw);return Member10(name,space,est,cfg['rep'],True,cols=cols,scaler=scaler)

# A missing feature must contribute no Bernoulli presence or absence evidence.
toy=sparse.csr_matrix([[1,0],[1,0],[0,1],[0,1]],dtype=np.float32);toy_m=sparse.csr_matrix(toy.shape,dtype=np.float32)
nbtoy=MissingBernoulli10(alpha=1.).fit(toy,np.array([0,0,1,1]),toy_m)
assert np.allclose(nbtoy.predict_proba(sparse.csr_matrix([[1,0]],dtype=np.float32),sparse.csr_matrix([[0,0]],dtype=np.float32)),[[.9,.1]])
assert np.allclose(nbtoy.predict_proba(sparse.csr_matrix([[0,0]],dtype=np.float32),sparse.csr_matrix([[1,1]],dtype=np.float32)),[[.5,.5]])
assert all(kind10(t)=='synonymous' for t in expand10('1_2FS>FS'))
(OUT/'configurations_v10.json').write_text(json.dumps(C10,indent=2));print('V10 candidates:',len(C10),'semantic checks passed')

C10.update({'functional_svm_balanced': {'rep': 'functional', 'algo': 'svm', 'C': 0.1, 'k': 2048, 'mask': 0.0, 'balanced_calibration': True}, 'functional_svm_balanced_mask10': {'rep': 'functional', 'algo': 'svm', 'C': 0.1, 'k': 2048, 'mask': 0.1, 'balanced_calibration': True}, 'type_complement_nb_a0.1': {'rep': 'type', 'algo': 'cnb', 'alpha': 0.1, 'k': 4096, 'mask': 0.0}, 'type_complement_nb_a1': {'rep': 'type', 'algo': 'cnb', 'alpha': 1.0, 'k': 4096, 'mask': 0.0}, 'functional_complement_nb_a1': {'rep': 'functional', 'algo': 'cnb', 'alpha': 1.0, 'k': 4096, 'mask': 0.0}})
(OUT/'configurations_v10.json').write_text(json.dumps(C10,indent=2))
print('Extended candidates:',len(C10))


V10 candidates: 19 semantic checks passed
Extended candidates: 24


In [8]:
# P43: HGVS 2016 principles; this is a conservative dataset parser, not a full HGVS validator.
AA16=set('ACDEFGHIKLMNPQRSTVWY');THREE16=dict(zip(['Ala','Arg','Asn','Asp','Cys','Gln','Glu','Gly','His','Ile','Leu','Lys','Met','Phe','Pro','Ser','Thr','Trp','Tyr','Val','Ter','Xaa'],list('ARNDCQEGHILKMFPSTWYV')+['*','X']))
# Each event: key, kind, position, components(position,ref,alt,kind), flags, fs_stop_distance.
def parse_token16(raw):
 s=str(raw).strip();core=s[2:] if s.startswith('p.') else s;uncertain=core.startswith('(') and core.endswith(')');core=core[1:-1] if uncertain else core
 if re.match('^(?:'+'|'.join(THREE16)+')[0-9]+',core):core=re.sub('|'.join(THREE16),lambda m:THREE16[m[0]],core)
 key=('uncertain|' if uncertain else '')+core;flags=['uncertain'] if uncertain else [];pos=None;parts=();dist=None;kind='unparsed'
 def effect(a,b):
  if a=='*' and b=='*':return 'stop_retained'
  if a=='*':return 'stop_lost' if b in AA16 else 'ambiguous_residue'
  if b=='*':return 'stop_gained' if a in AA16 else 'ambiguous_residue'
  if a not in AA16 or b not in AA16:return 'ambiguous_residue'
  return 'synonymous' if a==b else 'missense'
 m=re.fullmatch(r'([A-Z*])([1-9][0-9]*)([A-Z*])',core)
 if m:
  a,p,b=m.groups();pos=int(p);kind=effect(a,b);parts=((pos,a,b,kind),)
  if 'X' in [a,b]:flags.append('ambiguous_X')
 else:
  m=re.fullmatch(r'([1-9][0-9]*)_([1-9][0-9]*)([A-Z*]+)>([A-Z*]+)',core)
  if m:
   start,end,ref,alt=m.groups();start=int(start);end=int(end);pos=start;span=end-start+1
   if span>0 and span==len(ref)==len(alt) and all(a in AA16 or a=='*' for a in ref+alt):
    parts=tuple((start+j,a,b,effect(a,b)) for j,(a,b) in enumerate(zip(ref,alt)));kind='synonymous_block' if all(a==b for a,b in zip(ref,alt)) else 'block_change';flags.append('resolved_equal_length_block')
   else:kind='complex_range';flags.append('unresolved_range')
  elif 'fs' in core:
   kind='frameshift';m=re.match(r'([A-Z*]+)([1-9][0-9]*)',core);pos=int(m[2]) if m else None;m=re.search(r'fs(?:\*|Ter)([1-9][0-9]*)$',core);dist=int(m[1]) if m else None
  elif 'delins' in core:kind='delins'
  elif 'del' in core:kind='deletion'
  elif 'ins' in core:kind='insertion'
  elif 'dup' in core:kind='duplication'
  elif 'ext' in core:kind='extension'
  elif core=='=':kind='no_change_unspecified'
  elif re.fullmatch(r'[A-Z*]+[1-9][0-9]*',core):kind='incomplete';flags.append('missing_alternate')
  if pos is None:
   # Protein coordinate only: do not mistake fs termination distance for the affected residue.
   m=re.match(r'(?:[A-Z*]+)?([1-9][0-9]*)',core);pos=int(m[1]) if m else None
 return (key,kind,pos,parts,tuple(flags),dist)

def parse_rows16(frame):
 cols=list(frame.columns);cache={};rows=[]
 for row in frame.itertuples(index=False,name=None):
  mutations=[];missing=[];raw_n=0;duplicates=0
  for gene,value in zip(cols,row):
   if value is None or pd.isna(value) or (isinstance(value,str) and not value.strip()):missing.append(gene);continue
   if str(value).strip()=='WT':continue
   tokens=str(value).split();raw_n+=len(tokens);unique=list(dict.fromkeys(tokens));duplicates+=len(tokens)-len(unique);events=[]
   for token in unique:
    if token not in cache:cache[token]=parse_token16(token)
    events.append(cache[token])
   mutations.append((gene,tuple(events)))
  rows.append({'mutations':tuple(mutations),'missing':tuple(missing),'raw_tokens':raw_n,'duplicate_tokens':duplicates})
 return rows

FIXTURES16=[('R132H','missense',132),('R132R','synonymous',132),('R213*','stop_gained',213),('*363*','stop_retained',363),('*213Q','stop_lost',213),('K16Rfs*23','frameshift',16),('1_2FS>FS','synonymous_block',1),('483_484MP>IA','block_change',483),('10_12AAA>DD','complex_range',10),('490del','deletion',490),('p.(Arg123His)','missense',123),('R123X','ambiguous_residue',123),('A391','incomplete',391),('FS','unparsed',None),('G12delinsAA','delins',12)]
for token,expected_kind16,expected_pos16 in FIXTURES16:
 result=parse_token16(token);assert result[1]==expected_kind16 and result[2]==expected_pos16,(token,result)
assert parse_token16('K16Rfs*23')[5]==23
assert parse_token16('1_2FS>FS')[3]==((1,'F','F','synonymous'),(2,'S','S','synonymous'))
assert 'uncertain' in parse_token16('p.(Arg123His)')[4]
assert parse_rows16(pd.DataFrame({'G':['WT',None,'R1H R1H']}))[1]['missing']==('G',)
assert parse_rows16(pd.DataFrame({'G':['WT',None,'R1H R1H']}))[2]['duplicate_tokens']==1
print('P43 annotation safety fixtures passed:',len(FIXTURES16),flush=True)
# P25-P40: small, explicit gene subsets in the publications, not whole pathways or diagnostic rules.
PANELS16={
 'P25_breast':('TP53','PIK3CA','GATA3','MAP3K1'),
 'P26_colorectal':('APC','TP53','SMAD4','PIK3CA','KRAS','ARID1A','SOX9','FAM123B','POLE','MLH1'),
 'P27_glioma':('IDH1','IDH2','TP53','ATRX','CIC','FUBP1','NOTCH1'),
 'P29_gbm':('EGFR','PDGFRA'),
 'P30_renal':('VHL','PBRM1','SETD2','ARID1A','SMARCA4'),
 'P31_papillary_renal':('MET','SETD2','FH'),
 'P32_ovarian':('TP53','NF1','BRCA1','BRCA2','RB1','CDK12'),
 'P33_uterine':('TP53','PTEN','CTNNB1','PIK3CA','ARID1A','KRAS','ARID5B','POLE'),
 'P34_thyroid':('BRAF','EIF1AX','PPM1D','CHEK2'),
 'P35_prostate':('SPOP','FOXA1','IDH1'),
 'P36_lung_ad':('RIT1','MGA','EGFR','RBM10','NF1','MET','ERBB2'),
 'P37_lung_scc':('TP53','HLA-A','NFE2L2','KEAP1','CDKN2A','RB1'),
 'P38_melanoma':('BRAF','NRAS','HRAS','KRAS','NF1','KIT'),
 'P39_gastric':('PIK3CA','RHOA'),
 'P40_headneck':('PIK3CA','TRAF3','TP53','CDKN2A','HRAS','CASP8','NOTCH1','NSD1','AJUBA','FAT1','NFE2L2')}
print('Literature panel subsets:',len(PANELS16),'(external biological prior; no external patient data)',flush=True)


P43 annotation safety fixtures passed: 15
Literature panel subsets: 15 (external biological prior; no external patient data)


In [9]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_selection import chi2
from sklearn.preprocessing import MaxAbsScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from scipy import sparse
from collections import Counter
import copy
VIEWS16=['legacy_raw','hierarchical','rare_backoff','rare_backoff_panels']
# Literature-derived panel snapshots are explicit priors, not learned class labels.
class Preprocess16:
 def __init__(self,view,min_df=3,k=4096):self.view=view;self.min_df=min_df;self.k=k
 def fit(self,rows,target):
  self.df=Counter((g,e[0]) for row in rows for g,evs in row['mutations'] for e in {e[0]:e for e in evs}.values())
  records=self.records(rows,training=True);self.vec=DictVectorizer(dtype=np.float32);a=self.vec.fit_transform(records);names=self.vec.get_feature_names_out();support=np.asarray((a!=0).sum(0)).ravel();keep=(support>=self.min_df)|np.array([s.startswith(('num:','module:','numeric:','panel:')) for s in names]);self.keep=np.flatnonzero(keep);a=a[:,self.keep]
  with warnings.catch_warnings():
   warnings.simplefilter('ignore');scores=np.nan_to_num(chi2(a,target)[0],nan=0,posinf=0,neginf=0)
  self.cols=np.sort(np.argsort(-scores,kind='stable')[:min(self.k,a.shape[1])]);self.names=names[self.keep[self.cols]];selected=a[:,self.cols];self.scale=MaxAbsScaler().fit(selected);self.residual_weights=np.ones(len(self.names),np.float32)
  if self.view.startswith('rare_backoff'):
   for j,name in enumerate(self.names):
    if name.startswith('variant:'):
     _,gene,key=name.split(':',2);n=self.df.get((gene,key),0);self.residual_weights[j]=n/(n+5.)
  return self
 def records(self,rows,training=False):
  # legacy records were computed pointwise using the frozen v6 raw encoder.
  if self.view=='legacy_raw':return [r['legacy'] for r in rows]
  result=[]
  for row in rows:
   d={};types=Counter();gene_effects={};n=0;rare=0;new=0;uncertain=0;incomplete=0;components=0;ambiguous_x=0
   for g,evs in row['mutations']:
    evs=list({e[0]:e for e in evs}.values());d['gene:'+g]=1.;d['gene_event_count:'+g]=np.log1p(len(evs));local=Counter();bin64=set();bin256=set()
    for key,kind,pos,parts,flags,fsdist in evs:
     n+=1;support=max(0,self.df.get((g,key),0)-(int(training)));israre=support<3;rare+=israre;new+=support==0;uncertain+='uncertain' in flags;incomplete+=kind in ['incomplete','unparsed','complex_range'];ambiguous_x+='ambiguous_X' in flags
     effects=[p[3] for p in parts] or [kind];types.update(effects);local.update(effects);d['format:'+g+':'+kind]=1.;components+=len(parts)
     if self.df.get((g,key),0)>=3:
      d['variant:'+g+':'+key]=1.
     if self.view.startswith('rare_backoff'):
      bucket='rare' if israre else 'supported';d[bucket+'_gene:'+g]=d.get(bucket+'_gene:'+g,0)+1
     for p,a,b,effect in parts:
      d['aa_meta:'+g+':'+a+'>'+b]=d.get('aa_meta:'+g+':'+a+'>'+b,0)+1;d['aa_global:'+a+'>'+b]=d.get('aa_global:'+a+'>'+b,0)+1
     if pos is not None:bin64.add((pos-1)//64);bin256.add((pos-1)//256)
     # Absolute protein bins are an engineering heuristic, not genomic bins or biological domains.
     if fsdist is not None:d['fs_stop_distance:'+g]=np.log1p(fsdist)
    for kind,count in local.items():d['effect:'+g+':'+kind]=np.log1p(count)
    for b in bin64:d[f'protein_bin64:{g}:{b}']=1.
    for b in bin256:d[f'protein_bin256:{g}:{b}']=1.
    gene_effects[g]=local
   for name,count in [('observed_genes',len(row['mutations'])),('unique_events',n),('raw_tokens',row['raw_tokens']),('duplicate_tokens',row['duplicate_tokens']),('missing_genes',len(row['missing'])),('uncertain_events',uncertain),('incomplete_events',incomplete),('ambiguous_X',ambiguous_x),('resolved_components',components)]:d['num:log_'+name]=np.log1p(count)
   for kind in ['synonymous','missense','stop_gained','stop_lost','stop_retained','frameshift','deletion','insertion','delins','complex_range','incomplete','unparsed','ambiguous_residue']:
    d['num:log_'+kind]=np.log1p(types[kind]);d['num:fraction_'+kind]=types[kind]/max(sum(types.values()),1)
   if self.view.startswith('rare_backoff'):d['num:log_rare_events']=np.log1p(rare);d['num:rare_fraction']=rare/max(n,1);d['num:log_unseen_events']=np.log1p(new)
   if self.view.endswith('panels'):
    observed=set(gene_effects)
    for panel,gs in PANELS16.items():
     count=len(observed.intersection(gs));d['panel:'+panel+':mutated_genes']=np.log1p(count)
     flags=sum(bool(set(gene_effects.get(g,{})).intersection({'stop_gained','frameshift','deletion','delins'})) for g in gs);d['panel:'+panel+':truncation_notation_genes']=np.log1p(flags)
   result.append(d)
  return result
 def transform(self,rows,training=False):
  a=self.vec.transform(self.records(rows,training=training));a=a[:,self.keep][:,self.cols];return sparse.csr_matrix(self.scale.transform(a).multiply(self.residual_weights),dtype=np.float32)

class Model16:
 def __init__(self,view,algo):self.view=view;self.algo=algo
 def fit(self,rows,target):
  self.pre=Preprocess16(self.view).fit(rows,target);a=self.pre.transform(rows,training=True)
  self.est=LogisticRegression(C=.1,class_weight='balanced',max_iter=1200) if self.algo=='lr' else ExtraTreesClassifier(n_estimators=200,max_depth=20,min_samples_leaf=2,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=16016)
  self.est.fit(a,target);return self
 def predict_proba(self,rows):
  p=self.est.predict_proba(self.pre.transform(rows));out=np.zeros((len(rows),NC),np.float64);out[:,self.est.classes_.astype(int)]=p;return out
 def predict_frame(self,frame):
  rows=parse_rows16(frame[GENES16]);legacy=EvidenceEncoder(1.,1.).fit([]).transform(parse_events(frame[GENES16]))
  for row,d in zip(rows,legacy):row['legacy']=d
  return self.predict_proba(rows)


In [10]:
# Protocol frozen before new outer results. The old holdout is never scored.
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight
import gc
GENES16=list(genes)
parsed17=parse_rows16(X)
raw17=EvidenceEncoder(1.,1.).fit([]).transform(parse_events(X))
for row,record in zip(parsed17,raw17):row['legacy']=record
# Identical canonical and raw profiles stay in the same group.
parent17=np.arange(len(train))
def root17(i):
 while parent17[i]!=i:parent17[i]=parent17[parent17[i]];i=parent17[i]
 return i
def join17(a,b):parent17[root17(b)]=root17(a)
canonical17=[hashlib.sha256(repr((tuple(sorted((g,tuple(sorted(set(e[0] for e in ev)))) for g,ev in row['mutations'])),tuple(sorted(row['missing'])))).encode()).hexdigest() for row in parsed17]
for identifiers in [groups,canonical17]:
 seen17={}
 for i,key in enumerate(identifiers):
  if key in seen17:join17(seen17[key],i)
  else:seen17[key]=i
G17=np.array([root17(i) for i in range(len(train))]);overlap17=set(G17[dev])&set(G17[hold]);D17=dev[~np.isin(G17[dev],list(overlap17))]
assert not set(G17[D17])&set(G17[hold])
PLANS17={'v10_reference':(1.,0.,0.),'add_lr20':(.8,.2,0.),'add_lgb20':(.8,0.,.2),'add_both40':(.6,.2,.2)}
PROTOCOL17={'version':17,'outer_seed':17017,'outer_folds':3,'inner_seed_base':17018,'inner_folds':2,'plans':PLANS17,'new_view':'rare_backoff','k':8192,'min_df':3,'lr_C':1.,'lgb':{'n_estimators':300,'num_leaves':15,'max_depth':5,'learning_rate':.04,'reg_lambda':10.,'min_child_samples':20},'selection':'inner mean Macro F1; baseline wins exact tie','test_opened':False,'holdout_scored':False,'adoption_gate':'selected-v10 mean >=0.003 and every outer delta>=0; else experimental only','historical_development_bias':'same development cohort used before; new seeds do not make external validation'}
SHA17=hashlib.sha256((DATA/'train.csv').read_bytes()).hexdigest()
# Frozen baseline parameters are embedded; no baseline search.
def f17(target,p):return float(f1_score(target,p.argmax(1),average='macro',labels=np.arange(NC),zero_division=0))
class Rich17:
 def __init__(self,algo):self.algo=algo
 def fit(self,rows,target):
  self.pre=Preprocess16('rare_backoff',min_df=3,k=8192).fit(rows,target);a=self.pre.transform(rows,training=True)
  if self.algo=='lr':self.est=LogisticRegression(C=1.,class_weight='balanced',max_iter=1600)
  else:self.est=LGBMClassifier(n_estimators=300,num_leaves=15,max_depth=5,learning_rate=.04,reg_lambda=10.,min_child_samples=20,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=17017)
  self.est.fit(a,target);return self
 def predict_proba(self,rows):
  p=self.est.predict_proba(self.pre.transform(rows));out=np.zeros((len(rows),NC));out[:,self.est.classes_.astype(int)]=p;return out
 def predict_frame(self,frame):return self.predict_proba(parse_rows16(frame[GENES16]))
class Ensemble17:
 def __init__(self,old,weights,nb,rich,plan):self.old=old;self.weights=weights;self.nb=nb;self.rich=rich;self.plan=plan
 def predict_frame(self,frame):
  w=PLANS17[self.plan];p=w[0]*(.75*predict_members(self.old,self.weights,frame)+.25*self.nb.predict_frame(frame))
  for key,weight in [('lr',w[1]),('lgb',w[2])]:
   if weight:p+=weight*self.rich[key].predict_frame(frame)
  return p



In [11]:
from xgboost import XGBClassifier
from sklearn.ensemble import ExtraTreesClassifier
import gc
# Freeze a small algorithm-diversity experiment. Previous split is development, not fresh validation.
PLANS18={'v17_reference':(1.,0.,0.),'xgb20':(.8,.2,0.),'extra20':(.8,0.,.2),'both20':(.8,.1,.1)}
PROTOCOL18={'version':18,'outer_seed':17017,'outer_folds':3,'inner_seed_base':17018,'inner_folds':2,'confirmation_seed':18028,'confirmation_folds':3,'plans':PLANS18,'baseline':'fixed v17 add_both40','view':'rare_backoff','k':8192,'min_df':3,'xgb':{'n_estimators':350,'max_depth':4,'learning_rate':.04,'min_child_weight':3,'reg_lambda':10,'subsample':.9,'colsample_bytree':.8,'max_bin':63},'extra':{'n_estimators':400,'max_depth':24,'min_samples_leaf':2,'max_features':.15},'primary_gate':'nested mean delta>=0.003 and each fold>=0','confirmation_gate':'fixed-plan mean delta>=0.003 and each fold>=0','test_opened':False,'holdout_scored':False,'historical_bias':'previous development patients and primary split already inspected; confirmation seed is also same patients'}
(OUT/'protocol.json').write_text(json.dumps(PROTOCOL18,indent=2))
class New18:
 def __init__(self,algo):self.algo=algo
 def fit(self,rows,target):
  self.pre=Preprocess16('rare_backoff',min_df=3,k=8192).fit(rows,target);a=self.pre.transform(rows,training=True)
  if self.algo=='xgb':
   self.est=XGBClassifier(n_estimators=350,max_depth=4,learning_rate=.04,min_child_weight=3,reg_lambda=10,subsample=.9,colsample_bytree=.8,max_bin=63,tree_method='hist',objective='multi:softprob',eval_metric='mlogloss',n_jobs=2,random_state=18018)
   self.est.fit(a,target,sample_weight=compute_sample_weight('balanced',target))
  else:
   self.est=ExtraTreesClassifier(n_estimators=400,max_depth=24,min_samples_leaf=2,max_features=.15,class_weight='balanced',n_jobs=2,random_state=18018);self.est.fit(a,target)
  return self
 def predict_proba(self,rows):
  p=self.est.predict_proba(self.pre.transform(rows));out=np.zeros((len(rows),NC));out[:,self.est.classes_.astype(int)]=p;return out
 def predict_frame(self,frame):return self.predict_proba(parse_rows16(frame[GENES16]))
class Ensemble18:
 def __init__(self,baseline,models,plan):self.baseline=baseline;self.models=models;self.plan=plan
 def predict_frame(self,frame):
  w=PLANS18[self.plan];p=w[0]*self.baseline.predict_frame(frame)
  for name,weight in [('xgb',w[1]),('extra',w[2])]:
   if weight:p+=weight*self.models[name].predict_frame(frame)
  return p

def baseline18(tr,va,tag):
 path=WORK/'artifacts/v17'/(tag+'.npz');meta=path.with_suffix('.json')
 if path.exists() and meta.exists():
  info=json.loads(meta.read_text());assert info['sha']==SHA17 and info['protocol']==json.loads(json.dumps(PROTOCOL17))
  with np.load(path) as q:
   assert np.array_equal(q['tr'],tr) and np.array_equal(q['va'],va)
   p=.6*q['base']+.2*q['lr']+.2*q['lgb']
  return p
 ms,ws=fit_v6(tr);nb=train10('type_complement_nb_a0.1',tr,{})
 p=.6*(.75*predict_members(ms,ws,X.iloc[va])+.25*nb.predict_frame(X.iloc[va]));del ms,nb;gc.collect()
 for algo in ['lr','lgb']:
  model=Rich17(algo).fit([parsed17[i] for i in tr],yi[tr]);p+=.2*model.predict_proba([parsed17[i] for i in va]);del model;gc.collect()
 return p

def predict_fold18(tr,va,tag,oldtag=None):
 assert not set(G17[tr])&set(G17[va]);assert not set(G17[tr])&set(G17[hold]);assert not set(va)&set(hold)
 path=OUT/(tag+'.npz');meta=path.with_suffix('.json')
 if path.exists() and meta.exists():
  info=json.loads(meta.read_text());assert info['sha']==SHA17 and info['protocol']==json.loads(json.dumps(PROTOCOL18))
  with np.load(path) as q:
   assert np.array_equal(q['tr'],tr) and np.array_equal(q['va'],va);return {key:q[key] for key in ['base','xgb','extra']}
 started=time.time();p={'base':baseline18(tr,va,oldtag or tag)}
 print(tag,'baseline ready',round(time.time()-started,1),flush=True)
 for algo in ['xgb','extra']:
  model=New18(algo).fit([parsed17[i] for i in tr],yi[tr]);p[algo]=model.predict_proba([parsed17[i] for i in va]);print(tag,algo,'F1',round(f17(yi[va],p[algo]),5),'seconds',round(time.time()-started,1),flush=True);del model;gc.collect()
 for value in p.values():assert np.isfinite(value).all() and np.allclose(value.sum(1),1)
 np.savez_compressed(path,tr=tr,va=va,**p);meta.write_text(json.dumps({'sha':SHA17,'protocol':PROTOCOL18},indent=2));return p

def mix18(p,plan):
 w=PLANS18[plan];return w[0]*p['base']+w[1]*p['xgb']+w[2]*p['extra']
outer18=list(StratifiedGroupKFold(3,shuffle=True,random_state=17017).split(X.iloc[D17],yi[D17],G17[D17]));inner18=[];chosen18=[];rows18=[];oof18={key:np.zeros((len(D17),NC),np.float32) for key in [*PLANS18,'nested_selected']}
for fold,(a,b) in enumerate(outer18):
 tr=D17[a];va=D17[b];scores={key:[] for key in PLANS18}
 for j,(ia,ib) in enumerate(StratifiedGroupKFold(2,shuffle=True,random_state=17018+fold).split(X.iloc[tr],yi[tr],G17[tr])):
  it=tr[ia];iv=tr[ib];p=predict_fold18(it,iv,f'outer{fold}_inner{j}')
  for plan in PLANS18:
   value=f17(yi[iv],mix18(p,plan));scores[plan].append(value);inner18.append({'outer_fold':fold,'inner_fold':j,'plan':plan,'macro_f1':value})
 winner=max(PLANS18,key=lambda key:np.mean(scores[key]));chosen18.append({'fold':fold,'plan':winner,'inner_f1':float(np.mean(scores[winner]))});print('FROZEN INNER WINNER',chosen18[-1],scores,flush=True)
 p=predict_fold18(tr,va,f'outer{fold}')
 for plan in PLANS18:
  prediction=mix18(p,plan);oof18[plan][b]=prediction;rows18.append({'fold':fold,'plan':plan,'macro_f1':f17(yi[va],prediction),'accuracy':accuracy_score(yi[va],prediction.argmax(1))})
 oof18['nested_selected'][b]=mix18(p,winner);rows18.append({'fold':fold,'plan':'nested_selected','macro_f1':f17(yi[va],oof18['nested_selected'][b]),'accuracy':accuracy_score(yi[va],oof18['nested_selected'][b].argmax(1))})
 pd.DataFrame(inner18).to_csv(OUT/'inner_results.csv',index=False);pd.DataFrame(chosen18).to_csv(OUT/'inner_selected.csv',index=False);pd.DataFrame(rows18).to_csv(OUT/'outer_results.csv',index=False);np.savez_compressed(OUT/'oof_probabilities.npz',dev=D17,labels=labels,**oof18)
 print('OUTER',fold,pd.DataFrame(rows18).query('fold==@fold').to_string(index=False),flush=True)
summary18=pd.DataFrame(rows18).groupby('plan')[['macro_f1','accuracy']].mean();delta18=[f17(yi[D17[b]],oof18['nested_selected'][b])-f17(yi[D17[b]],oof18['v17_reference'][b]) for a,b in outer18];votes18=Counter(r['plan'] for r in chosen18);choice18=max(PLANS18,key=lambda key:(votes18[key],np.mean([r['macro_f1'] for r in inner18 if r['plan']==key])))
metrics18={'version':18,'train_sha256':SHA17,'baseline_mean_f1':float(summary18.loc['v17_reference','macro_f1']),'nested_mean_f1':float(summary18.loc['nested_selected','macro_f1']),'fold_deltas':delta18,'mean_delta':float(np.mean(delta18)),'primary_gate_passed':float(np.mean(delta18))>=.003 and min(delta18)>=0,'final_plan_inner_votes':choice18,'inner_votes':dict(votes18),'test_opened':False,'sample_opened':False,'holdout_scored':False,'submission_created':False,'model_saved':False,'forbidden_attempts':forbidden_attempts18}
summary18.to_csv(OUT/'comparison.csv');(OUT/'metrics.json').write_text(json.dumps(metrics18,indent=2));print('V18 PRIMARY',summary18.to_string(),metrics18,flush=True)
# New confirmation seed is evaluated only when the primary gate passes. Never retune the plan here.
confirmation18=[]
if metrics18['primary_gate_passed'] and choice18!='v17_reference':
 confirm_p18={key:np.zeros((len(D17),NC),np.float32) for key in ['baseline','candidate']}
 for fold,(a,b) in enumerate(StratifiedGroupKFold(3,shuffle=True,random_state=18028).split(X.iloc[D17],yi[D17],G17[D17])):
  p=predict_fold18(D17[a],D17[b],f'confirmation18028_fold{fold}');cp=mix18(p,choice18);fb=f17(yi[D17[b]],p['base']);fc=f17(yi[D17[b]],cp);confirmation18.append({'fold':fold,'baseline_f1':fb,'candidate_f1':fc,'delta':fc-fb});confirm_p18['baseline'][b]=p['base'];confirm_p18['candidate'][b]=cp;pd.DataFrame(confirmation18).to_csv(OUT/'confirmation_results.csv',index=False);print('FIXED CONFIRMATION',confirmation18[-1],flush=True)
 np.savez_compressed(OUT/'confirmation_probabilities.npz',dev=D17,labels=labels,**confirm_p18)
 metrics18['confirmation_mean_delta']=float(np.mean([r['delta'] for r in confirmation18]));metrics18['confirmation_gate_passed']=metrics18['confirmation_mean_delta']>=.003 and min(r['delta'] for r in confirmation18)>=0
else:metrics18['confirmation_gate_passed']=False;metrics18['confirmation_skipped']='primary gate did not pass'
metrics18['local_improvement_confirmed']=bool(metrics18['primary_gate_passed'] and metrics18['confirmation_gate_passed'])
if metrics18['local_improvement_confirmed']:
 path17=WORK/'artifacts/v17/cancer_model_v17_experimental.pkl'
 if path17.exists():
  proof17=json.loads((path17.parent/'verification.json').read_text());assert proof17['train_sha256']==SHA17 and hashlib.sha256(path17.read_bytes()).hexdigest()==proof17['model_sha256']
  with path17.open('rb') as f:base_final18=cloudpickle.load(f)
 else:
  all18=np.arange(len(train));ms,ws=fit_v6(all18);nb=train10('type_complement_nb_a0.1',all18,{});rich={key:Rich17(key).fit(parsed17,yi) for key in ['lr','lgb']};base_final18=Ensemble17(ms,ws,nb,rich,'add_both40')
 w=PLANS18[choice18];models18={key:New18(key).fit(parsed17,yi) for key,weight in [('xgb',w[1]),('extra',w[2])] if weight};final18=Ensemble18(base_final18,models18,choice18)
 with (OUT/'cancer_model_v18_experimental.pkl').open('wb') as f:cloudpickle.dump(final18,f)
 pd.DataFrame([{'model':key,'feature':name} for key,m in models18.items() for name in m.pre.names]).to_csv(OUT/'new_selected_variables.csv',index=False);metrics18['model_saved']=True
assert not forbidden_attempts18
(OUT/'metrics.json').write_text(json.dumps(metrics18,indent=2));print('V18 FINISHED',metrics18,flush=True)


outer0_inner0 baseline ready 0.0
outer0_inner0 xgb F1 0.39165 seconds 23.2
outer0_inner0 extra F1 0.39254 seconds 31.4
outer0_inner1 baseline ready 0.0
outer0_inner1 xgb F1 0.37907 seconds 21.7
outer0_inner1 extra F1 0.3746 seconds 29.4
FROZEN INNER WINNER {'fold': 0, 'plan': 'xgb20', 'inner_f1': 0.42899197165703973} {'v17_reference': [0.4331832250673777, 0.40246832965487], 'xgb20': [0.4436132330473865, 0.41437071026669303], 'extra20': [0.44094017411793557, 0.40638677932098666], 'both20': [0.4418871043059453, 0.41466128898449156]}
outer0 baseline ready 0.0
outer0 xgb F1 0.4227 seconds 29.8
outer0 extra F1 0.40474 seconds 45.1
OUTER 0  fold            plan  macro_f1  accuracy
    0   v17_reference  0.470057  0.468845
    0           xgb20  0.469665  0.471869
    0         extra20  0.466408  0.468240
    0          both20  0.465958  0.471264
    0 nested_selected  0.469665  0.471869
outer1_inner0 baseline ready 0.0
outer1_inner0 xgb F1 0.35713 seconds 23.2
outer1_inner0 extra F1 0.37594 

## 검증 증거·저장 모델 재로딩·분석 보고서

In [12]:
from pathlib import Path
import subprocess,tempfile,importlib.metadata as metadata18
# Validate actual fold membership and saved probabilities.
audit18=[]
for p18 in OUT.glob('*.npz'):
 with np.load(p18,allow_pickle=False) as q18:
  if 'tr' not in q18.files or 'base' not in q18.files:continue
  t18=q18['tr'];v18=q18['va'];assert not set(G17[t18])&set(G17[v18]);assert not set(t18)&set(hold);assert not set(v18)&set(hold)
  for key18 in ['base','xgb','extra']:assert np.isfinite(q18[key18]).all() and np.allclose(q18[key18].sum(1),1)
  audit18.append({'cache':p18.name,'train_rows':len(t18),'validation_rows':len(v18),'group_overlap':0,'old_holdout_rows_used':0})
pd.DataFrame(audit18).to_csv(OUT/'fold_audit.csv',index=False)
if metrics18['model_saved']:
 replay18="""import sys,os,cloudpickle,pandas as pd,numpy as np
from pathlib import Path
def guard(e,a):
 if e=='open' and a and isinstance(a[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(a[0])).name in {'test.csv','sample_submission.csv'}:raise RuntimeError('Inference input forbidden')
sys.addaudithook(guard)
with open(sys.argv[1],'rb') as f:m=cloudpickle.load(f)
x=pd.read_csv(sys.argv[2]).iloc[:12].drop(columns=['ID','SUBCLASS']);p=m.predict_frame(x)
assert p.shape==(12,26) and np.isfinite(p).all() and np.allclose(p.sum(1),1)
assert np.allclose(m.predict_frame(x.iloc[3:8]),p[3:8])
np.save(sys.argv[3],p)
"""
 with tempfile.TemporaryDirectory() as tmp18:
  probe18=Path(tmp18)/'probe.npy';proc18=subprocess.run([sys.executable,'-c',replay18,str((OUT/'cancer_model_v18_experimental.pkl').resolve()),str((DATA/'train.csv').resolve()),str(probe18)],cwd=tmp18,capture_output=True,text=True);assert proc18.returncode==0,proc18.stderr;assert np.allclose(np.load(probe18),final18.predict_frame(X.iloc[:12]))
 metrics18['independent_train_reload']='passed';metrics18['model_sha256']=hashlib.sha256((OUT/'cancer_model_v18_experimental.pkl').read_bytes()).hexdigest();metrics18['model_bytes']=(OUT/'cancer_model_v18_experimental.pkl').stat().st_size
assert not forbidden_attempts18
metrics18['audited_fold_caches']=len(audit18);metrics18['status']='verified';(OUT/'metrics.json').write_text(json.dumps(metrics18,indent=2));(OUT/'verification.json').write_text(json.dumps(metrics18,indent=2))
# Pooled paired group bootstrap is descriptive for fixed OOF only, not an additional selection rule.
rng18=np.random.default_rng(18029);ug18=np.unique(G17[D17]);positions18={g:np.flatnonzero(G17[D17]==g) for g in ug18};bootstrap18=[]
for _ in range(500):
 idx18=np.concatenate([positions18[g] for g in rng18.choice(ug18,size=len(ug18),replace=True)]);bootstrap18.append(f17(yi[D17[idx18]],oof18['nested_selected'][idx18])-f17(yi[D17[idx18]],oof18['v17_reference'][idx18]))
metrics18['paired_group_bootstrap_pooled_delta_95pct']=np.quantile(bootstrap18,[.025,.975]).tolist();(OUT/'metrics.json').write_text(json.dumps(metrics18,indent=2));(OUT/'verification.json').write_text(json.dumps(metrics18,indent=2))
report18=f'''# v18 — v17 이후 트리 다양성 실험

## 목적과 고정한 조건

v17을 고정 기준으로 두고 추가 XGBoost·ExtraTrees가 예측 오류를 보완하는지 시험했습니다. [v17](analysis_report_v17.md)의 rare_backoff 표현과 동일하게 min_df 3, 학습 chi2 최대 8,192열, 학습 MaxAbs, 스케일 이후 exact DF 감쇠를 적용했습니다. [문헌 근거와 입력 한계](LITERATURE_REVIEW_AND_PREPROCESSING_V16.md)는 그대로 유지하며 새로운 외부 환자나 정답을 추가하지 않았습니다.

추가 XGBoost는 350회·depth 4·learning rate 0.04·min_child_weight 3·lambda 10·subsample 0.9·colsample 0.8·hist max_bin 63, 균형 sample weight를 사용했습니다. ExtraTrees는 400개·depth 24·min_leaf 2·max_features 0.15·balanced로 고정했습니다. 파라미터 검색으로 outer 점수를 최대화하지 않았습니다.

후보는 v17 단독, XGB 20% 추가, ExtraTrees 20% 추가, 두 모델 각 10% 추가의 네 개입니다. 모든 환자에게 동일한 확률 혼합을 적용하며 클래스별 예측 개수를 조정하지 않았습니다. 낮은 중요도 변수나 상관 계수는 validation을 보고 추가 제거하지 않았습니다.

## 데이터 경계와 검증

- 공식 train 6,201명 중 기존 개발 {len(D17):,}명으로 검증했습니다. test/sample은 open guard로 차단하고 기존 홀드아웃 1,241명은 점수를 계산하지 않았습니다.
- raw/정규화 동일 프로파일을 union group으로 묶고 학습·검증·기존 홀드아웃의 경계를 확인했습니다.
- 이전에 관찰한 outer seed 17017 그룹 3-fold, inner 2-fold seed 17018+fold를 사용했습니다. **이 분할은 개발용 재사용이며 새 독립 검증이 아닙니다.**
- v17 구성은 이 버전에서 고정했습니다. v17 캐시도 SHA256·프로토콜·학습/검증 인덱스가 맞아야 사용하며 없으면 해당 학습 환자로 재학습합니다. 전체-train 피클은 CV 예측에 사용하지 않았습니다.
- 추가 모델의 어휘·빈도·변수 선별·스케일·학습은 해당 학습 fold 안에서만 적합했습니다.
- 조합은 inner 평균 Macro F1로 선택하고 그 상태로 outer에서 평가했습니다. 전체 outer 최고 조합으로 다시 선택하지 않습니다.
- 사전 기준은 평균 delta +0.003 이상과 모든 fold 비하락입니다. 통과한 경우에만 새 seed 18028 그룹 3-fold에서 inner 다수결 조합을 고정해 확인하며, 확인 기준도 동일합니다.

## 실제 결과

{summary18.to_markdown()}

주 결과인 nested 선택의 평균 F1은 **{metrics18['nested_mean_f1']:.6f}**, 기준 v17은 **{metrics18['baseline_mean_f1']:.6f}**, 차이는 **{metrics18['mean_delta']:+.6f}**입니다. Fold별 차이 `{delta18}`. 고정 OOF의 그룹 bootstrap pooled delta 95% 구간은 `{metrics18['paired_group_bootstrap_pooled_delta_95pct']}`이며, 학습 무작위성 전체를 포함하지 않습니다.

inner 선택:

{pd.DataFrame(chosen18).to_markdown(index=False)}

최종 inner 다수결 조합 `{choice18}`. 1차 기준 통과 **{metrics18['primary_gate_passed']}**.

### 고정 조합 추가 확인

{pd.DataFrame(confirmation18).to_markdown(index=False) if confirmation18 else metrics18.get('confirmation_skipped','미수행')}

추가 기준 통과 **{metrics18['confirmation_gate_passed']}**. 두 단계의 로컬 개선 확인 **{metrics18['local_improvement_confirmed']}**.

표의 다른 조합 점수는 탐색적 대조이며 주 결과를 대체하지 않습니다. 0.60 달성·Public 향상은 확인되지 않았습니다. 과거 반복 개발/홀드아웃/test EDA의 설계 편향을 이 중첩 검증이 소급 제거하지 않습니다. 새 seed도 같은 개발 환자의 재분할이며 외부 검증이 아닙니다.

## 저장·재현·판단

새 모델 저장 **{metrics18['model_saved']}**. 두 단계 기준을 통과한 경우에만 전체 train으로 추가 모델을 학습하고, 전체 train으로 저장한 v17과 결합해 `artifacts/v18/cancer_model_v18_experimental.pkl`에 모든 전처리·모델을 포함해 저장합니다. 이 전체-train 모델로 기존 홀드아웃을 평가하면 누수입니다. 독립 프로세스 train-only 재로딩 **{metrics18.get('independent_train_reload','새 모델 미저장')}**.

기준 미통과 시 v17을 유지하고 새 모델·제출 CSV를 만들지 않습니다. 기준 통과도 실제 대회 성능을 보장하지 않으며 기존 주 모델·제출 파일을 자동 교체하지 않습니다. 원본 train/test 파일 변경, 신규 test 추론, 대회 제출, 커밋·푸시는 하지 않았습니다.

`cencer_v18.ipynb`에 파서·기준 구성·전처리·변수 선택·nested 학습·추가 확인·최종 저장·보고서 생성을 포함합니다. `requirements-v18.txt`와 공식 `data/train.csv`가 필요합니다. 프로젝트 .py와 test는 실행 의존성이 아닙니다. 캐시가 없으면 기준 모델도 학습하므로 오래 걸릴 수 있습니다. 실제 fold 예측·선택 기록·프로토콜·검증 증거는 `artifacts/v18/`에 보관하고 Git에서 제외합니다.
'''
(WORK/'analysis_report_v18.md').write_text(report18)
variable18='# v18 추가 모델 변수 기록\n\n표현·변수 의미는 [v17 변수](selected_variables_v17.md) 및 [v16 근거](LITERATURE_REVIEW_AND_PREPROCESSING_V16.md)를 따릅니다. 학습 min_df 3·chi2 최대 8,192열·MaxAbs·exact DF 감쇠를 고정하고 선택은 각 학습 fold 안에서만 수행했습니다. 전체 train 선택 목록을 검증 fold에 재사용하지 않았습니다.\n\n최종 모델 저장: **'+str(metrics18['model_saved'])+'**.\n'
if metrics18['model_saved']:
 families18=[]
 for algo18,model18 in models18.items():
  families18.extend({'model':algo18,'family':k,'selected_columns':v} for k,v in Counter(name.split(':',1)[0] for name in model18.pre.names).items())
 variable18+='\n전체 train 재학습의 추가 모델 열 수:\n\n'+pd.DataFrame(families18).to_markdown(index=False)+'\n\n실제 전체 변수 목록: `artifacts/v18/new_selected_variables.csv`. 중요도를 인과·병원성으로 해석하지 않습니다.\n'
else:variable18+='\n미채택 실험이므로 최종 전체-train 변수 목록을 생성하지 않았습니다. Fold별로 학습한 변수/모델은 검증용이며 배포 모델이 아닙니다. v17을 유지합니다.\n'
(WORK/'selected_variables_v18.md').write_text(variable18)
packages18=['numpy','pandas','scipy','scikit-learn','lightgbm','xgboost','cloudpickle','nbformat','ipython','threadpoolctl','tabulate'];(WORK/'requirements-v18.txt').write_text('# Python 3.15\n'+'\n'.join(p+'=='+metadata18.version(p) for p in packages18)+'\n')
print('FINAL V18',metrics18,flush=True)


FINAL V18 {'version': 18, 'train_sha256': '92418b8441d058cfc68e939dd88725610750be4bc8edc51253cffc72fc4fc0ab', 'baseline_mean_f1': 0.47007834419004113, 'nested_mean_f1': 0.47682967112510416, 'fold_deltas': [-0.00039193928359160246, 0.008335712153593133, 0.012310207935187556], 'mean_delta': 0.006751326935063029, 'primary_gate_passed': False, 'final_plan_inner_votes': 'xgb20', 'inner_votes': {'xgb20': 3}, 'test_opened': False, 'sample_opened': False, 'holdout_scored': False, 'submission_created': False, 'model_saved': False, 'forbidden_attempts': [], 'confirmation_gate_passed': False, 'confirmation_skipped': 'primary gate did not pass', 'local_improvement_confirmed': False, 'audited_fold_caches': 9, 'status': 'verified', 'paired_group_bootstrap_pooled_delta_95pct': [-0.002985559546442749, 0.014379377220806762]}


## 1차 기준 미통과를 유지한 별도 후속 진단

In [13]:
# Post-gate diagnostic. Preserve the original failed gate; never rename it as a pass.
FOLLOWUP18={'seed':18028,'folds':3,'plan':'xgb20','selection':'frozen unanimous original inner winners; no new search','original_primary_gate_remains':False,'diagnostic_signal':'mean delta>=0.003 and every diagnostic fold>=0','not_adoption':True}
(OUT/'followup_protocol.json').write_text(json.dumps(FOLLOWUP18,indent=2));follow_rows18=[];follow_oof18={key:np.zeros((len(D17),NC),np.float32) for key in ['baseline','candidate']}
for fold,(a,b) in enumerate(StratifiedGroupKFold(3,shuffle=True,random_state=18028).split(X.iloc[D17],yi[D17],G17[D17])):
 tr=D17[a];va=D17[b];assert not set(G17[tr])&set(G17[va]);assert not set(G17[tr])&set(G17[hold]);path=OUT/f'followup18028_{fold}.npz';meta=path.with_suffix('.json')
 if path.exists() and meta.exists():
  inf=json.loads(meta.read_text());assert inf['train_sha256']==SHA17 and inf['protocol']==FOLLOWUP18
  with np.load(path) as q:
   assert np.array_equal(q['tr'],tr) and np.array_equal(q['va'],va);bp=q['baseline'];cp=q['candidate']
 else:
  started=time.time();bp=baseline18(tr,va,f'followup18028_{fold}');m=New18('xgb').fit([parsed17[i] for i in tr],yi[tr]);cp=.8*bp+.2*m.predict_proba([parsed17[i] for i in va]);del m;gc.collect()
  for p in [bp,cp]:assert np.isfinite(p).all() and np.allclose(p.sum(1),1)
  np.savez_compressed(path,tr=tr,va=va,baseline=bp,candidate=cp);meta.write_text(json.dumps({'train_sha256':SHA17,'protocol':FOLLOWUP18},indent=2));print('FOLLOWUP18 elapsed',fold,round(time.time()-started,1),flush=True)
 follow_oof18['baseline'][b]=bp;follow_oof18['candidate'][b]=cp;fb=f17(yi[va],bp);fc=f17(yi[va],cp);follow_rows18.append({'fold':fold,'baseline_f1':fb,'candidate_f1':fc,'delta':fc-fb});pd.DataFrame(follow_rows18).to_csv(OUT/'followup_results.csv',index=False);print('FOLLOWUP18',follow_rows18[-1],flush=True)
np.savez_compressed(OUT/'followup_probabilities.npz',dev=D17,labels=labels,**follow_oof18)
metrics18=json.loads((OUT/'metrics.json').read_text());metrics18['followup_mean_delta']=float(np.mean([r['delta'] for r in follow_rows18]));metrics18['followup_baseline_mean_f1']=float(np.mean([r['baseline_f1'] for r in follow_rows18]));metrics18['followup_candidate_mean_f1']=float(np.mean([r['candidate_f1'] for r in follow_rows18]));metrics18['followup_diagnostic_positive']=metrics18['followup_mean_delta']>=.003 and min(r['delta'] for r in follow_rows18)>=0;metrics18['followup_is_not_original_gate_pass']=True;metrics18['followup_model_saved']=False
if metrics18['followup_diagnostic_positive']:
 base_path18=WORK/'artifacts/v17/cancer_model_v17_experimental.pkl'
 if base_path18.exists():
  proof17=json.loads((base_path18.parent/'verification.json').read_text());assert proof17['train_sha256']==SHA17 and hashlib.sha256(base_path18.read_bytes()).hexdigest()==proof17['model_sha256']
  with base_path18.open('rb') as f:base_final18=cloudpickle.load(f)
 else:
  ix=np.arange(len(train));ms,ws=fit_v6(ix);nb=train10('type_complement_nb_a0.1',ix,{});rich={key:Rich17(key).fit(parsed17,yi) for key in ['lr','lgb']};base_final18=Ensemble17(ms,ws,nb,rich,'add_both40')
 new18=New18('xgb').fit(parsed17,yi);follow_final18=Ensemble18(base_final18,{'xgb':new18},'xgb20');follow_path18=OUT/'cancer_model_v18_followup_experimental.pkl'
 with follow_path18.open('wb') as f:cloudpickle.dump(follow_final18,f)
 pd.DataFrame({'model':'xgb','feature':new18.pre.names}).to_csv(OUT/'followup_selected_variables.csv',index=False)
 import subprocess,tempfile
 replay="""import sys,os,cloudpickle,pandas as pd,numpy as np
from pathlib import Path
def guard(e,a):
 if e=='open' and a and isinstance(a[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(a[0])).name in {'test.csv','sample_submission.csv'}:raise RuntimeError('inference forbidden')
sys.addaudithook(guard)
with open(sys.argv[1],'rb') as f:m=cloudpickle.load(f)
x=pd.read_csv(sys.argv[2]).iloc[:12].drop(columns=['ID','SUBCLASS']);p=m.predict_frame(x);assert p.shape==(12,26) and np.isfinite(p).all() and np.allclose(p.sum(1),1);assert np.allclose(m.predict_frame(x.iloc[3:8]),p[3:8]);np.save(sys.argv[3],p)
"""
 with tempfile.TemporaryDirectory() as tmp:
  probe=Path(tmp)/'p.npy';proc=subprocess.run([sys.executable,'-c',replay,str(follow_path18.resolve()),str((DATA/'train.csv').resolve()),str(probe)],cwd=tmp,capture_output=True,text=True);assert proc.returncode==0,proc.stderr;assert np.allclose(np.load(probe),follow_final18.predict_frame(X.iloc[:12]))
 metrics18['followup_model_saved']=True;metrics18['followup_model_sha256']=hashlib.sha256(follow_path18.read_bytes()).hexdigest();metrics18['followup_independent_train_reload']='passed'
assert not forbidden_attempts18
(OUT/'metrics.json').write_text(json.dumps(metrics18,indent=2));(OUT/'verification.json').write_text(json.dumps(metrics18,indent=2))
extra_doc18='\n## 1차 기준 미통과 이후 고정 조합 후속 진단\n\n원래 채택 기준 미통과 상태를 유지한 별도 진단입니다. 세 inner fold가 모두 선택한 XGB 20% 조합을 고정하고, 원래 지정했던 seed 18028에서 학습·평가를 다시 했습니다. 새 조합/비율/파라미터는 탐색하지 않았습니다. 이 적응적 후속 진단으로 원래 실패한 기준을 통과로 바꾸지 않습니다. 같은 개발 환자의 재분할이며 외부 평가가 아닙니다.\n\n'+pd.DataFrame(follow_rows18).to_markdown(index=False)+f"\n\n후속 평균 delta **{metrics18['followup_mean_delta']:+.6f}**, 진단 신호 양성 **{metrics18['followup_diagnostic_positive']}**. 후속 실험용 모델 저장 **{metrics18['followup_model_saved']}**. 저장한 경우 파일은 `artifacts/v18/cancer_model_v18_followup_experimental.pkl`이며 원래 gate 통과 모델과 구분합니다. 전체 train으로 재학습했으며 위 검증에 사용하지 않았습니다. 기존 모델·제출 CSV를 교체하지 않았고 Public 점수는 미확인입니다.\n"
p=WORK/'analysis_report_v18.md';p.write_text(p.read_text()+extra_doc18)
p=WORK/'selected_variables_v18.md';p.write_text(p.read_text()+f"\n## 후속 진단의 별도 실험용 모델\n\n저장 여부: **{metrics18['followup_model_saved']}**. 저장 시 실제 목록은 `artifacts/v18/followup_selected_variables.csv`입니다. v17 80%+새 XGB 20%이며 동일 rare_backoff min_df 3·chi2 최대 8,192열을 사용합니다. 원래 gate 미통과를 뒤집는 모델 채택이 아닙니다.\n")
print('V18 FOLLOWUP FINISHED',metrics18,flush=True)


FOLLOWUP18 elapsed 0 139.4
FOLLOWUP18 {'fold': 0, 'baseline_f1': 0.45591946122955057, 'candidate_f1': 0.4690602305385905, 'delta': 0.013140769309039901}
FOLLOWUP18 elapsed 1 146.8
FOLLOWUP18 {'fold': 1, 'baseline_f1': 0.4630661705430502, 'candidate_f1': 0.4711518212401362, 'delta': 0.008085650697086}
FOLLOWUP18 elapsed 2 132.4
FOLLOWUP18 {'fold': 2, 'baseline_f1': 0.47023590426977674, 'candidate_f1': 0.47783481068262024, 'delta': 0.007598906412843498}
V18 FOLLOWUP FINISHED {'version': 18, 'train_sha256': '92418b8441d058cfc68e939dd88725610750be4bc8edc51253cffc72fc4fc0ab', 'baseline_mean_f1': 0.47007834419004113, 'nested_mean_f1': 0.47682967112510416, 'fold_deltas': [-0.00039193928359160246, 0.008335712153593133, 0.012310207935187556], 'mean_delta': 0.006751326935063029, 'primary_gate_passed': False, 'final_plan_inner_votes': 'xgb20', 'inner_votes': {'xgb20': 3}, 'test_opened': False, 'sample_opened': False, 'holdout_scored': False, 'submission_created': False, 'model_saved': False, 'for

## 과거 고정 스트레스 진단 — 재선택 없음

In [14]:
# Historical stress diagnostic: frozen recipes; no mixture or feature retuning.
from sklearn.naive_bayes import ComplementNB
from xgboost import XGBClassifier
CURRENT_STRESS_VERSION=int(OUT.name[1:]);assert CURRENT_STRESS_VERSION in [18,19]
STRESS_PROTOCOL={'seed':43,'v18_weights':[.8,.2],'v19_weights':[.9,.1],'xgb':{'trees':350,'depth':4,'lr':.04,'child':3,'lambda':10,'subsample':.9,'colsample':.8,'bins':63,'seed':18018},'cnb':{'alpha':1.,'temperature':2.},'preprocessor':'rare_backoff min_df3 chi2 k8192'}
STRESS_CACHE=WORK/'artifacts/v18_19_stress';STRESS_CACHE.mkdir(exist_ok=True);prior_path=WORK/'artifacts/v17/historical_stress_probabilities.npz';prior_proof=WORK/'artifacts/v17/verification.json';prior_predictions=None
if prior_path.exists() and prior_proof.exists():
 assert json.loads(prior_proof.read_text())['train_sha256']==SHA17
 with np.load(prior_path,allow_pickle=True) as previous:
  assert np.array_equal(previous['dev'],dev) and np.array_equal(previous['labels'],labels);prior_predictions=previous['candidate']
 assert prior_predictions.shape==(3,len(dev),NC)
stress_records=[]
for fold,(a,b) in enumerate(StratifiedGroupKFold(3,shuffle=True,random_state=43).split(X.iloc[dev],yi[dev],groups[dev])):
 tr=dev[a];va=dev[b];assert not set(G17[tr])&set(G17[va]);assert not set(G17[tr])&set(G17[hold]);cache=STRESS_CACHE/f'fold{fold}.npz';meta=cache.with_suffix('.json')
 if cache.exists() and meta.exists():
  info=json.loads(meta.read_text());assert info['sha']==SHA17 and info['protocol']==STRESS_PROTOCOL
  with np.load(cache) as q:
   assert np.array_equal(q['tr'],tr) and np.array_equal(q['va'],va);bp=q['base'];p18=q['candidate18'];p19=q['candidate19']
 else:
  frames=[X.iloc[va],rare_background(X.iloc[va],[events[i] for i in tr],42+901+fold),duplicate_frame(X.iloc[va])]
  if prior_predictions is not None:bp=prior_predictions[:,b]
  else:
   old,ow=fit_v6(tr);nb=train10('type_complement_nb_a0.1',tr,{});bp=np.array([.6*(.75*predict_members(old,ow,f)+.25*nb.predict_frame(f)) for f in frames]);del old,nb;gc.collect()
   for algo in ['lr','lgb']:
    old_rich=Rich17(algo).fit([parsed17[i] for i in tr],yi[tr])
    for j,f in enumerate(frames):bp[j]+=.2*old_rich.predict_frame(f)
    del old_rich;gc.collect()
  pre=Preprocess16('rare_backoff',min_df=3,k=8192).fit([parsed17[i] for i in tr],yi[tr]);at=pre.transform([parsed17[i] for i in tr],training=True)
  xm=XGBClassifier(n_estimators=350,max_depth=4,learning_rate=.04,min_child_weight=3,reg_lambda=10,subsample=.9,colsample_bytree=.8,max_bin=63,tree_method='hist',objective='multi:softprob',eval_metric='mlogloss',n_jobs=2,random_state=18018).fit(at,yi[tr],sample_weight=compute_sample_weight('balanced',yi[tr]));nm=ComplementNB(alpha=1.,norm=False).fit(at,yi[tr]);p18=np.zeros_like(bp);p19=np.zeros_like(bp)
  for j,f in enumerate(frames):
   av=pre.transform(parse_rows16(f));xp=np.zeros((len(f),NC));npred=np.zeros_like(xp);xp[:,xm.classes_.astype(int)]=xm.predict_proba(av);npred[:,nm.classes_.astype(int)]=softmax(nm.predict_log_proba(av)/2.,axis=1);p18[j]=.8*bp[j]+.2*xp;p19[j]=.9*bp[j]+.1*npred
  del pre,at,xm,nm;gc.collect();np.savez_compressed(cache,tr=tr,va=va,base=bp,candidate18=p18,candidate19=p19);meta.write_text(json.dumps({'sha':SHA17,'protocol':STRESS_PROTOCOL},indent=2))
 chosen_prediction=p18 if CURRENT_STRESS_VERSION==18 else p19
 for j,condition in enumerate(['clean','synthetic_rare_background','duplicate_tokens']):
  assert np.isfinite(chosen_prediction[j]).all() and np.allclose(chosen_prediction[j].sum(1),1);fb=f17(yi[va],bp[j]);fc=f17(yi[va],chosen_prediction[j]);stress_records.append({'fold':fold,'condition':condition,'baseline_f1':fb,'candidate_f1':fc,'delta':fc-fb})
 print('Historical frozen stress diagnostic',CURRENT_STRESS_VERSION,fold,flush=True)
stress_table=pd.DataFrame(stress_records).groupby('condition')[['baseline_f1','candidate_f1','delta']].mean();pd.DataFrame(stress_records).to_csv(OUT/'historical_stress_results.csv',index=False);stress_table.to_csv(OUT/'historical_stress_summary.csv')
proof=json.loads((OUT/'verification.json').read_text());proof['historical_stress_deltas']=stress_table['delta'].to_dict();proof['stress_diagnostic_not_selection']=True;assert not proof['test_opened'] and not proof['holdout_scored'];(OUT/'metrics.json').write_text(json.dumps(proof,indent=2));(OUT/'verification.json').write_text(json.dumps(proof,indent=2))
marker='\n## 과거 고정 스트레스 진단\n';p=WORK/f'analysis_report_v{CURRENT_STRESS_VERSION}.md';text=p.read_text().split(marker)[0];p.write_text(text+marker+'\n이전 seed 43의 개발 분할에 고정 조합을 적용한 추가 진단입니다. 해당 학습 fold의 rare background만 검증 입력에 합성 추가하고, 중복 표기도 확인했습니다. 학습은 원본 학습 환자로만 수행했고, v17 캐시는 train 해시와 개발 인덱스를 확인했습니다. 원본 train/test는 변경하지 않았습니다. 실제 test 분포나 Public 점수의 대용이 아니며, 이 결과로 비율·변수·파라미터를 다시 선택하지 않았습니다.\n\n'+stress_table.to_markdown()+'\n')
print(stress_table.to_string(),flush=True)


Historical frozen stress diagnostic 18 0
Historical frozen stress diagnostic 18 1
Historical frozen stress diagnostic 18 2
                           baseline_f1  candidate_f1     delta
condition                                                     
clean                         0.473020      0.478908  0.005888
duplicate_tokens              0.468270      0.472149  0.003879
synthetic_rare_background     0.358136      0.344012 -0.014125


## 최종 진단 판단 — 기준 변경 없음

In [15]:
noise_delta_final=float(json.loads((OUT/'verification.json').read_text())['historical_stress_deltas']['synthetic_rare_background'])
final_proof=json.loads((OUT/'verification.json').read_text())
final_proof['stress_diagnostic_priority']='defer: rare-background degradation' if noise_delta_final<-.003 else 'no degradation in checked historical conditions'
(OUT/'metrics.json').write_text(json.dumps(final_proof,indent=2));(OUT/'verification.json').write_text(json.dumps(final_proof,indent=2))
final_note='\n## 최종 진단 판단\n\n'
if noise_delta_final<-.003:
 final_note+='희귀 변이 추가 조건에서 기준 대비 '+format(noise_delta_final,'+.6f')+' 하락했습니다. 이 진단으로 원래 기준을 바꾸지는 않지만, 높은 원본 평균만으로 이 후보를 우선 제출하도록 권하지 않습니다. 후속 피클은 실험 기록용으로 보존하고 v17 또는 더 일관적인 v19 후보를 우선 비교합니다.\n'
else:
 final_note+='확인한 희귀 변이 추가·중복 조건에서도 하락하지 않았습니다. v19는 사전 원본 기준과 추가 분할 기준을 통과한 소폭 개선 후보입니다. 다만 OOF bootstrap 구간에 0이 포함되므로 큰 차이가 확립됐다고 주장하지 않으며 Public 향상은 미확인입니다. 기존 모델·제출 CSV는 유지합니다.\n'
p=WORK/('analysis_report_'+OUT.name+'.md');p.write_text(p.read_text()+final_note)
print(final_proof['stress_diagnostic_priority'])


defer: rare-background degradation


## 선택 실행: 사용자 승인 최종 제출 추론
이 셀부터는 test/sample을 **최종 추론에만** 사용합니다. 앞선 문서·지표의 test 미사용은 학습·선별·검증 단계에 대한 기록입니다. 이 셀은 재학습·튜닝하지 않고 저장 모델로 제출 CSV를 만듭니다. 연구용 open guard는 유지하고 별도 추론 프로세스를 사용합니다. 기존 제출 파일과 결과가 다르면 덮어쓰지 않습니다. 저장 모델이 있으면 이 셀만 별도 실행할 수 있습니다. v18 파일은 스트레스 조건 성능 저하가 있는 후속 실험용이며, v19와 구분하세요.

In [ ]:
# 사용자 승인 최종 제출 추론: 학습/검증 단계와 구분합니다.
from pathlib import Path
import subprocess,sys
roots_export=[Path.cwd(),*Path.cwd().parents]
repo_export=next((p for p in roots_export if (p/'data'/'test.csv').is_file()),None)
if repo_export is None:raise FileNotFoundError('공식 data/test.csv와 sample_submission.csv가 필요합니다.')
work_export=repo_export/'models'/'혜진' if (repo_export/'models'/'혜진').is_dir() else repo_export
model_export=work_export/'artifacts'/'v18'/'cancer_model_v18_followup_experimental.pkl'
if not model_export.is_file():raise FileNotFoundError('앞선 학습 셀에서 저장한 모델이 필요합니다.')
# 연구 단계 open guard가 설치된 부모와 구분해 새 추론 프로세스에서만 test를 읽습니다.
# fit/선별/파라미터 변경 없이 저장 모델의 predict만 호출합니다.
inference_program="from pathlib import Path\nimport sys,json,cloudpickle,numpy as np,pandas as pd,hashlib\nfrom threadpoolctl import threadpool_limits\nthreadpool_limits(2)\nmodelpath,testpath,samplepath,outpath,labelpath=map(Path,sys.argv[1:6])\nwith modelpath.open('rb') as f:model=cloudpickle.load(f)\ntest=pd.read_csv(testpath);sample=pd.read_csv(samplepath)\nassert list(sample.columns)==['ID','SUBCLASS'] and test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)\nx=test.drop(columns=['ID'])\nif isinstance(model,dict):\n classes=np.asarray(model['classes']);predict=lambda frame:model['predict'](model,frame)\nelse:\n with np.load(labelpath,allow_pickle=True) as q:classes=np.asarray(q['labels']).astype(str)\n predict=model.predict_frame\np=np.asarray(predict(x));assert p.shape==(len(test),len(classes)) and len(classes)==26 and np.isfinite(p).all() and (p>=-1e-8).all() and np.allclose(p.sum(1),1,atol=1e-5)\nassert np.allclose(predict(x.iloc[:8]),p[:8],atol=1e-6)\nresult=sample.copy();result.SUBCLASS=result.ID.map(pd.Series(classes[p.argmax(1)],index=test.ID))\nassert result.ID.equals(sample.ID) and not result.isna().any().any()\npayload=result.to_csv(index=False).encode('utf-8');outpath.parent.mkdir(exist_ok=True,parents=True)\nif outpath.exists():\n assert outpath.read_bytes()==payload,'기존 제출 파일과 다릅니다. 파일을 확인하고 별도 이름을 지정하세요.'\nelse:outpath.write_bytes(payload)\nassert pd.read_csv(outpath).equals(result)\nproof={'rows':len(result),'test_use':'inference only','training_performed':False,'sample_id_order':'verified','probabilities':'verified','batch_invariance':'passed','model_sha256':hashlib.sha256(modelpath.read_bytes()).hexdigest(),'csv_sha256':hashlib.sha256(payload).hexdigest()}\nproofpath=modelpath.parent/'submission_inference_verification.json';proofpath.write_text(json.dumps(proof,ensure_ascii=False,indent=2));print(str(outpath));print(json.dumps(proof,ensure_ascii=False,indent=2))\n"
result_export=subprocess.run([sys.executable,'-c',inference_program,str(model_export.resolve()),str((repo_export/'data/test.csv').resolve()),str((repo_export/'data/sample_submission.csv').resolve()),str((repo_export/'submission'/'혜진_submission_v18_followup_experimental.csv').resolve()),str((work_export/'artifacts/v18/oof_probabilities.npz').resolve())],capture_output=True,text=True)
if result_export.returncode:raise RuntimeError(result_export.stderr)
print(result_export.stdout)
